# Kaggriculture: High-Performance Adaptive Agent

This notebook builds, inspects, and packages a Kaggriculture competition
agent centered on a pre-computed action route with a defensive adaptive
layer on top.

The route itself is a frozen, step-by-step plan compressed and embedded
directly into `main.py`. On its own, a frozen plan like this would fail the
moment the live board differs from the one it was recorded against — a
different tile layout, a different hand count, a weed where the plan
expected an empty tile, and so on. To make the route safe to submit, six
small adaptive functions sit around it at runtime, each one repairing a
specific way the plan and the live board can disagree.

The notebook is organized in three parts:

1. **Route inspection** — decode the embedded route and break down what it
   actually does: action frequency, daily activity, and market behavior.
2. **Market cross-validation** — pull the real pricing implementation
   straight from the installed competition environment and check that the
   route's trading behavior is consistent with each commodity's actual
   price sensitivity.
3. **Adaptive layer audit and packaging** — document what each defensive
   function protects against, then compile, hash, and archive `main.py`
   into the final `submission.tar.gz`.

**If you found this notebook useful and decide to fork it, please consider
leaving an upvote — it genuinely helps and is much appreciated!**

## The agent: `main.py`

The cell below writes the actual submission file. It embeds the compressed
action route (`_ACTIONS`) alongside the six runtime safety layers described
in the notebook overview — weed repair, sell-timing shift/preempt, sell-slot
ranking, terminal liquidation, hand-count alignment, and a top-level
`try/except` fallback to `PASS` on any unexpected error.

In [ ]:
%%writefile main.py

# Load the public adaptive route and runtime controller
import base64
import copy
import json
import math
import zlib

_ACTIONS = json.loads(zlib.decompress(base64.b85decode('c-rk<O>bM*5&bV(b74}HEO)2aOe{pP3`s7L8bT1DDGC(nBJHl|f3IRu<l~!}GiPS*eWcv1Oj9J^{l4>Y=A6&}Ir-bqzy12lZzq5HeDd+~?(XDacJlY1|M}N{J-+ey^4Fh#`^PW;etiA;<o(s{>hZ7Ki*G*t_|xTws~;|}Pi7}?Z`LQXg?Rh!{c81b@Q1tA>fPhp>-Ve6`;*!0(c3?)u5Uh^%;wvVf4seU_v!8Z?i*(h5C5I*_2=sC_n$uQo-`ly?eoccb$9=ztsib~@85rUwS8;!Vt*j+R@c|Nr_Rl%Za*-5>h`b0LAknq_tWFx-+$VS9@h?a5JYn}KcO{YH!Sueb7KG=y7|g!|DKP3ebAcSlq-`zerx#b@mybDzFloca_<qcZ`wn}EAX&yhx=oHa5v5PeNFxTTaW+$aKGI%`a6*)zr7p};HWK+Lv?w-x*ffGe(3H~qh_Fy9i2tnMhsiNy1X-<9{T0o56WTMK4Sag?&gy-T=EIZLf^J{`@?OAYrH0!kD6Elvi-_4pI+o9e%D?!W0gUZ$Isv}D2>);)iBdM8-6-5UTCq&&Dmz)#s^`C5hT`|d<R@3nRSPTFLN$z-WjxW_m1_b+yj)s+Wu+u$Yc+8?G-=#^dj)P=%c{A0$+Pxh0JHIi#BjWqL1EOU$5TY|MczZ_WtVn>MvhLt#ucsXwTTl10Q`p&;Dq7x#^Lu^2bM^N4s%i22(Iw+U{(?-`sp)3H{B;P7i(C_7iG0Km0c9l;L$Bvk`g?Q=|=Ym>PJlEeA=u;=D~H&c4{A?GfJDw{9Dh0Y*(|HN-n7$!nl8z=1JRhB)A9c3;EN{b)77gC&qKJIC&G(#xCr$puq8a&;x{rPwjs;1bGQJw4W8?lvx*efblw%T<y--h%gr_Z;^qOB`Sw@7~ae`3Jdu)24>ZJ(?z6V`2CIYx+vo<ruvvxf)D9Wt@GB+uE~UUP|nT3CG3x!ER>yteF>RM^_7Xk`ZEme|h`o_?<O2@ztpPrYY$-iDE_u#{^mLyWbusGBS6c5lAjgn_O1g$x2V07O!_h?S&cC&Zv@qt95{P=b+E4t+s+?Gn}m55AMAipFcTQAj9ZeCaFWNl<3(yNiz4!tY|7nVJ_`tcIEknGNZ*`v=TQ}6J&F#EzZ6y1J5jx7Q(#0ms#-{(&ygM_?T1s*zc~s>tz1ZJFJnfS&5qw2cl)_bc~XP#_Xwqu_4pik%J<&aH-hQ`<ng6sa(yMmcuG}f>U^X5%0r3_79rE0@m_zq_6@hl41^PXV9fGXjBR;6W;{xN88YF*^Bm<G2t?YXQ+jH)@q76+6QASXEK#l!>+Z-Wvy>+9{-uYLjK6_gZ5}mzAKa;Y`mDe``gR)H>=y*A0MCj#>99m9x^VQV$h7dF1C&$aYri^1~P0YU2FM*K3Nt-uz1{Nqg@iaD%%yn=ksQS9;Z|Q%z>x-_6P2I@aYZxH3K=$jnK2dH`ronlTm<vwI_3>7LgmtUN=e=LU;(22~Eu-u-lYaIF6K~i=@}W(pm&cf{Z~;FE?qlfU1^VTP2cz^yFL|RCCHN$7diMr(oWKoz+m*$ht+s)7JpC<cwEf_BtAHN|Awf*qiyoq<OA3OPx_;nA}}pyj{wlDLF;Aq1t@9jR-i_#(vZOiVpoik(#uc;8?fVdij7HsQ&3Td&tv~g*Gr<SVJjVt^t@5@9*eqt=guP@~BgGASeuW^yUFyPWmLkV_7E3g|)=l`2&y8-uLC^*)nDOIFm(nvb-u=uPiTUY0!}7IXk27ZL42<MZ^(;AmH<_;)x7X&11`2DWgx=Q(|#Q`xu}Xx>`%y7sxL66ONm87%+@f1V}B;uhp5Z*m7w(Yr}QHn|Z|gaOQ?t9P0=)$*o+OQD7zXgw1!<n6I{nBjy0%UOLBwJ3zAe5NgJQpeX3!-T_3H`ogUsELyW%IW{c#6y|?g%{3@U$y^n*7B&SIWMx2_{4QnR{cv^t{f5nFL>lwji1eR_t-x5geC`LPP2vjP@Z0PD+_6>1(5e}GVaf|k+K#&YS|h$e2LyG@*$pWMh+%-1G-T_cv7Mn}+`tTKvUPaawSLVs!&HEIa|uBxeZ-!Y8)na5x-_#+Hl95*64XlOmrl+qRqh2uvZO>+ch4=E{j?dLT^A*jaNBRi>Cm{8c$Z7|d-p7U^366Ry9u(zR6P;jNw92t*!B`eu6uSldmmMDgLO;9A+zWvd75n>_66)as{*OEKWD5~`CNsesm<I<^K}ppkWtJMZiY9WYw^<Hlk$f9WDf$sp9|?BoK_tH)+1N|0L`4E2#qplKKTX9B3EjH6xFBZT;#k5ru&H+(4aSwXay>PBfSY57}Fz)&0307?75ywMnj~QBSz{>8@OWn;OMPkm{X~5CTnG>AS!6BQ-9VA$%9Y~Yi5)gj^RR3g!Zjgz{TNsQwtuTvXj<$^8~h0dNcvfMzy2yq6}}Irt1J)D=3i^JPjH(Ut1yivQ@SpcC9R)<6O%Z51I6VHsC1M;J0wWk1fpnXk+*8_U0ppoYu{~p`n%@R~pA9R=4!T(4e;+Y5;GrU5o8_Z(oS{K?A*5Mh5z{O`DMq`qYi++BmHY^MGz6*Ji(tAuyycQW0w4c13{pwHZ@2-Nz*PqHc3xT-*BcRO<pG$fXF0!;z`_VZHwrBUaExVc~3s&p)hOTAVh(w840X@<+cSXXapbjR3QY51cQ}m6t-*4FyofZ!XMF#`<poTLF}%08e)Dd`s=kh54q^MI3dJX%K*@DS`*3=m4&9aS%OJM&OX(L@~LbS`Gj!8T7tp3`eGqgG6qpRFRK}&!2}SXCdZo!%sel(#{gG;CWaV(FRp0k(V+pN|gZxmYx|xB4%jWX!jX!Q+g+Mz=&Cm!Bxz|l~GtEblBh^84C4M7=|Jc$VLW%0#^*nVefP$!CfHICiWtjHaJOMg-~`iDgj=4=Pvs0TqQjoT$eMB(XmvcjUs4DX(g!KK?s$=<v)5>Y;S5msHEpn7JW6<Le2C~lcpM754*!Bgc%4gxe^#y_9NmL)29kp!0oxfqGJ#Gm0o0Bdl%bCpnFcrpb5HUEL2%*vA4d=p~<zrwmrPk&JL?<vMmy-7;U-{+Qs2YfiydUqJ+|hp_o`T!7vzhp%)J@fp_Hq*4k!+5!#T{0e{yv-b+xMh;eN<!2=IuT6WsX>UA0!u1C%1pFAwHjuN5@4)ROPE-{EiUvi-nwbFnBeM}heV4*UB^;IGO76XQhCW}#owNp7OM#smtv?;c<k0rqOUaKH=h62jtmUQWq2$}~l=h2>A(X8V5Qqjmj7~Br~!f0ATzd##L?MzD6Iiqr0Zvq41swCly6~GmiubS5{5n~cQK&HMq_r%to-2YWUf=oj1!xUbB1;~WM_&>3#Go%};IYY~b0K3>OtN<H|DnbeBwi;Wh94<SZ2o;zzm3#z44CTS0Th__Nq$z1>W|^?t2T3=B1Ku16nsEm)on2#oL}JayaS<%|!9HPXyTCT>5Z4dO3BZr`gu4K8RRTCbAumYEm{c3jPD?(+i!P<si$69MIliROs%Oub{P3E3ta>&$%$cPhDbR%~lqqPF%HK?S34D}-B;(2>KqwHXbE)gI%T=2^J6Z^N9##a}BjB;x`6d|zm)qI%ch_y@>15&vS1h`92i5kl-B%_1eE+$<U%v7=5~g>pmD-U&wZ?S8iz&hZp7Vcx7<PhbGmje{vqd|Hn#nFUb(ups#e%#>#7pj)Q*3Ge2Z(d~WD`Bp-pD4}KKZoXY5)cvF})(rs7yI>vgFM&*3s1!@Yd4erIm$-gC`*C*`Rbblb_@W4BBQVO?!EIcSs)hEs(kh_aI9p*fgeZ3@h{EJx1}o1_jC+MzB;AK$e$-q~;KZUzN6o=3ksqu?s-6oWnVE3;uY}*O}*hvgyx_Jm;Hk>}wNNtPkjyzt+5GGq|U2Do|vgEpnWZTp}=@iQ?U2(v!sV;nY*3Jm-<1{@{|WjXz^wQEq6L3|plTIPEE7zoU72Q`*gxC)s|mTP=1%A*FwoGXmQS3WvlOd?TLW$kJ8Lf+lHV#%c5%CAhxBujtGr0rOCnU9kLhMO5ae<nJI3)Q7ZY${uSx6p)FI;xcHHlcX|{YL-gL{z;1Zz%Ey~fn`F7y|xhRHb{fD-RUTyB~zD51uLLOD_9A@1=1So<uqf7)>0-jNh;ULX+Eyjl(LkQ;+Z#Y@%*Sg`>aEMe0f#n8$$L}{mVK23kQNeoi=ByHuu!<NBB~uZkb+0=Y|xUk0l?B+W&&H#{2WR`TqW@YL?b*pKTt%^UbC+!0}^CQm;7<@}cYMP2H5)ZDosq24<My4we<i$VMn~UJ_nNu#~g=A50Cit-xpaie#+Z>&KV4{QKUk?2053_uBd7Zh5ecl;fb}3PX}ed5I7G<z7oAQ{<~0O7$>fcBN2K%Lpk=gBD03NhKUm5$u+y3MMaA8q$4<NXIB%*8w(WwsV2J9p(1Qu!$k}0kGnMR43zZQWA74iC(EKh!!o+aVjwlYt_OkR}{LSr$tgshqjo%YF9meR7eXGCI5mVHp~kv9pFV_79C2_DYkv$R*ov(14D3l;A2|qHtn;WU7guv_%6UG5(3!C1`|GQfXWzqctJY+uE8PYh3Cj}#e<f&RY0=y@t`hdJ;ZW|;b{~zrKHi;1XWOMO5(c%?%;UB-ag=aQdZYV204Z*xJiz%=sqfQkZ>W}R@;NYM?@@+YGA04x#4DTK}{joTsz6>^=%(``YtGsC{G1I#__aQikcnP>l~^*F>FT0fB=jS$|NCmoRVC-%g+X&Fv{>V_hcE5Cu0hB(@+2mRa^5CEePzQ0+W%pCJHBzd>F~t5m$~6@HhyeQ;KR4jfeq0#(Ur0@AFqR8u+?WA+m+?NR&NsPBU(>fAZ-vj(Z`7vxCJ-K)h#R^l0%CC;}j&fZ-<1212p~<>dzV<u2!%*N23l1+^F1>|<s<$GPWHXZlg1EBCY>-HJe?Go<Oz_eKdp4pNmXy~_GMB)@y-r5oOkoHv=Jkr#^?sl~M+1=K^vm7K8aU9i#vOdUc}X;1AwMThOow(B`SU<h-K!72%THvP>G$@tmWaaOf03jiW2$q<EHX5U79YsDGnDqM`TeI>>Q$tRF&rL;5Pd&)b+1-2+L8tD)@xGo@Q{p1(CF<hYVwW9bWBU5?4*C_<s01z6`l~c=`d#K<wr2nnmCJ0|rL=UL&ea7=2E#_h~YNBI>G4aC_idVk()~@X2D5>$yMLg`}>C5EwLxEcWWk}TlbD#y?&;oh5sEgUMUW7@3HgB=Ks&pq$T0_v-IlR*jQ@ClAI~ab9cA(6vm5DQ@YT9WuemfL{xNcSGju$}5!A`ZV!JJIVCCwMOa9alpp<;A@3M~*Rb4iL_Q%5fF%~L_hGztP!#wZQ}!U&`$Y2E<%kgBa9UD!4=l)yrOat!ErVoXYbBZI<5WVa1I3Y5IQH!q5bdG-)MQZQg2SW6k5he+^9@tk42Ar6DXqxwu}7YHo{Y(QmEBgBLru=5F+HiM|nKB#s?Vh<#kiW}S+yEBo-flt;-3qGR7j?geUzg*3!D^J$Y2pr;yF;7F@uqw&Q6w1gdolfmaGXM7*vMQ6uIC9uB>~gxSA|GBFkH^u4S!{|6o8Lns8Nx&#It)+y(isoOn4k94NTg7PNG*szdn%PGAr6gZE$+mH%7l3Wn27bNMZzAWxpyD&i!O5%6K2L2CMY|zP$p}`VcI4_9e`4MVd9Q{iUurH3grt4k@i4FIIwNq7HEMpJ?hJ|SumRLjH{Q0-zaYo8#J^68N$tK&3=144`PNU^W7-c-5BPhpf7T+8dHxRm@+St-avN+VxIV8D!{xDCi93>=5&B7IrtU0rqv(JAsHb=CucdeTC=F=B0{c<$$XT`B?N(_XmemNcmMo1f1^}sD6b#s_$RrKQOFdgtrYH%1UgFfRIF&rOjp;5%p3UhK>z1Cc5BBEbw_@6ee*$62Fx&LY0*<9C_^s4?EvojCdSs?yX8(Z!mbkRWL;-8wa<!SV9FIripDD0)Z|i?{a2!dILDGv+VzltD8lEm*x=dJVvfjyF|~W1;8x30EDr^NQL*3}a6GlHA=~V|g@l^T7lt_N-k2?q9gwz%p?oFUzxetPX`puAN({#-juR?K1U(s7_tKESCc@Dg;)<w2)iB0On5CGl#wpL3rqZktFDuK&<M=}*=e~`|spos@_8IC|GgO-gkMcJSU<EY1Zye5JOp{CnUev<_{Q|WNZTvM|D6QzSK~xl8cwLx$$OT6t+AybQ3b6w*UOZHdL*7apR00sSG%+R&fQ{Dy&XYwo-Z~n#P`XsHo#PPksYe+|lEz%bxcC%ub5WW~S$dF<Kv{0*fx2xiK!G7{a*r#(NnQ<6j<pq#&FKj80%N9i8FR)7f)cI%3OJmcR8y0dL5^NbzY=DHx$r$uWuMCjI|w!qp^z?pW?bbJp$xFyh#;CRR=&8;(87ECRNN#6eB1uy(|ENd3kvZmku*Yl5m|sP`3*q`WCw(nMm%VOSkHc+nvgpmGZ~PLh(k=HK`7kWK}eQizfrQrgaQUG%t#gE+Gl17b|G`chvcqoumnb@K}K4t)knbO8g)9f?uSGO^0KpK+9f@x1p$>*G3B)7gC1=RNjC)7aYmt=!=2@neQ>1KPc2VdI7o1>vHQ+&pc5;w#RiL$LZNb{btHRkeWp>QW8F}Sor4ADq?EEGJYFarQ_66hycLOOuYKUy;DR>M;f}1k2<Bw?bl>bikTq>bwOL891GKCd&N;;p8JR^!GK_h^Q9890RJ|%JFem<|+^8^LkwxM)T5S^WsXG0Di<^VSDKQYofJ(5<6Vyp+LdscNF~pDmp#tJu;pTP@0uZKPU`sL0t`ZS=RZEQeP$pd2TULyjlMd2j!(iGQ875o2yoypsfNX&xi7zi2he#ADk?^dJ+5EhxkQpiDkn2vvg3)M$Il1;})y|?+O%>kqDP^AY?!UYGE>r;97d2AXfAT8|QL@~N%uO5I$dI-_&!=z7X{#WshsD9sDtc0S`dCb*cE~(;?)hK&t>?ca%xgQw2zKX+^yF|~164QG>6GL<I*5&Sz)_y%c{Z>v9S%5}`OZOdo22WJw85Eb*OUN*C+DW$Q9Rl&P=zidLEz$)m?MYf+dGRu&Cxz&Y1Tj6!Q8r}ty4Lf@{%VSmWh>igBiw_$5jDv>lhi*g5~w>)U73N8ZykDWB-@5a*EZhfNGR24EPYB*s6+CH_<~z1(wn`NQQ4Se!HPlrHIoyeausdgd1Hvgy@SIJ--p;i4^4Cu0{O~pHGWFOO=_VHyLE7WBLh@{MMgRwIVpejSqZJpNMALwF2s~BLtL{s-QLqwLnUHin>A;dn|`PraP$nLsXm1n76-&T&;(Me#9Fc&v(qFoBMcO)foR85P(rSx}^*832F#lbkq=J&SJ9PbGnm#bq7JaHS;IJb9ZQQLJt4OhylMC2}5{G;Ud%Z3Kp3jf!1NzLhu=&4y}vmLCQQvI<5OLwez}=;wloCaY|#sj}evU6R9l5rQOQ}c7ErXxK-x(aN%ekJ~AN4R$+kj91-S7cTyw)q&w(|#gt{&)YfUbsG}pv^rPJ`S`LmRhJytLJO&6Qo|Rwrp|b$}R$%7~Rck{z3x%!AyBsvoE6H4d_D4j1LFSS)W;M+p(F7uyNM495)fue}W=JXUQ*s7Ki4ouaD1yXHknV4vJ!JSGZJ%)-I<grOMM{??G*ikUW?Ol7WZNqRmXXczNa|TVp54d;0BS*TP-v}&$1wI5#77eG7lPFQwO=W4wGR$+NQ!-sfo)I*j-q@}ij)jTEf%Q|-G*81l>!m8&XE@trCYwt2pkiNW9TKc;M&<k7tv2fWjNaWu9~0Nb!gM#yyhpCVI*}j*%00#B|l-fi{po*%($xLCoV{H;&LtOS|`-|)C~n6gL*WwpI*w6Ls2qljss|YDkp7%yRQ<23vC<+ezIm<S$}&=XQsY25SH8D3^7Gw;6>A`o*ikl!UGw3(B$MAw;?)&kbrv-xy=Xcb}*79zo$`Ohhi-w>K~bE=Jh?W;wB1;A$5b;iPB8Mfn)S{h#WEnu5R!eT|!>91y^#NB<s^C>YQi-<U`iz+H%3r94Q;d#6a?`_<i20u#Ir>W5MxO4U^bFsv~jaQ>z9m*o;tX^FXH8GDtE%Gs+I<7snnYNwLizksjv~t7AeGlKq1wM4~?tnTCGW+PU!RP7JcP=}IM!T0Rph79a|T40BUNA2USS!@!}AU+j{^21l6D4yRg4MGPBBxwX`&q{tuG_*Nn!npB;{NHQ;P-jejHpOPdsA<&l-bJV8fw<;bPU2_Qnphm?JuG2&^|C1=cByAC3{c<bHD^yBOC^ZISBx%+nA?>=x?7Bs<seD1>dKi^I0$GS)NBP`MNJdkIhlGL~ijnOB3Q%;Jfutz;xWLkoqpk!5V1=UcSOPi`)$HV*P6kKAP){)cjxppW*J35!qHVqqah6%2RUK4Ny%i(HavlE(r3G_=icFlws?XY>Tok_evlpcnxcusB&a^b)ir_|d0WyA8<+(0+6xmt!(HhKlG{}42U48H{{U{3(Vw3|c-o`gIA8R?7M4`v<#ft$3wD6|YF67urS)M8Ly=Vxvu3JcF@=*$rdB>n`=`L@G=FrRXM+>%^RnldH9bTOk;DBI)3lr4U(QN0O@>^7cGvtSBK6G3s6blPr2vFuAL9rZ#A)I4MgryCAjt3<!A^|h8jpS(Pdh8_WLxOOfbf`STFaz{?R6msgh&T(`=Pu78Oi)3gesTkR6oMw@Re1qY$~L%m?l`I_g}7iuhX>CGl1}CHArkZRj4ZX-3kFK{&d54OK^g~+6ur{rHL<HCB#%}gb_xd33ZM!a(E%g!{229z_984&q;6-n$~RS2DYv0gdTdkuM^&*M;RBQ|iQxnlCMGXNt$YP)E3C!tvZ94Vc~p!Z48*Wd$4Sviz}{7j3_kndMg&dSe#Z;oI(Eu>=7oPiG(c4O@S|?!XKR9%ol|Q(4n;=r>R4qL+0v~-Hx8hrXt*{9sJcq1B?5NINdysBWZ_g`^L3Ci^oqIQ-<wo5(7J?F@m8tsB3VhqdRrhi7g~`>oW+-NAOw6vrgT{I4~87T@tJH@7!fA}UyPb!E((wotdyB>9HsZnT7~c2wq0cQ9VAFcAC4915D4j+19M3YhinQFkSF53sG&w=tp#^TO!N%6VEx)b`hC5d(FHwn3~gYg+)M<>v(IzU8v4;+%q>!U#Gl;g(no)D&&p&Ri{axxM7zM1kZg``e5(2J>`+0?xHUz58yGIkmkga7!WM`bDP*3M@FEiOXhE@Du8=2#PH|Viv)Tf10xpdhF^r)WN^)A0dLW%9*DpEpBk<{JMbA%0l9CLaPcr4LG{A!g26#s^ZS&nysFv=K;UbNl1agCMXTYC|@?l8cGg$VT5HeLF;ar0-hw{qn322hk(ZiQB$#DyXN@MzXmXd@Qu+!QMTdM2>nU~{wVM|U);(>EN7A>2x26=g$#g<t<zay_D0$<^ooL@Amr<7?#Nz2N0a{O`6g&B2-P@S{{g*+@Wpc5Un-h~&O>YL@^e?UVC3;')).decode('utf-8'))
__version__ = 'adaptive-preempt-3x2x1'

_PRICE_FLOOR = 1
_DEMAND_ALPHA = 0.25
_MARKET_PARAMS = {'WHEAT': (25, 10000, 400, 'sqrt', 0.8, 'log', 0.2), 'CARROT': (35, 10000, 450, 'log', 0.2, 'sqrt', 0.7), 'TOMATO': (60, 10000, 200, 'linear', 0.4, 'sqrt', 0.6), 'STRAWBERRY': (120, 10000, 100, 'sqrt', 0.7, 'linear', 1.6), 'MELON': (250, 10000, 300, 'log', 0.2, 'sq', 3.6), 'EGG': (50, 10000, 332, 'linear', 0.4, 'log', 0.2), 'MILK': (160, 10000, 122, 'sqrt', 0.6, 'linear', 1.6), 'WOOL': (200, 10000, 105, 'log', 0.2, 'sq', 3.2), 'FERTILIZER': (100, 10000, 200, 'linear', 0.4, 'linear', 0.4)}
_SHOP_PRODUCTS = {'BAKERY': ('EGG', 'WHEAT'), 'PIZZA_SHOP': ('MILK', 'TOMATO', 'WHEAT'), 'BRUNCH_SPOT': ('EGG', 'WHEAT', 'STRAWBERRY'), 'YARN_STORE': ('WOOL',), 'ICE_CREAM_SHOP': ('STRAWBERRY', 'MILK', 'WHEAT'), 'PET_CAFE': ('CARROT',), 'SMOOTHIE_SHOP': ('STRAWBERRY', 'MILK'), 'FARMERS_MARKET': ('WHEAT', 'CARROT', 'TOMATO', 'STRAWBERRY')}
_SELLABLE = tuple(_MARKET_PARAMS)
_LIQUIDATION_ORDER = ('CARROT', 'EGG', 'FERTILIZER', 'MELON', 'MILK', 'STRAWBERRY', 'TOMATO', 'WHEAT', 'WOOL')
_WEED_STATE = {0: {}, 1: {}}
_WEED_REPLAY_STEPS = 8
_SHIFT_STATE = {0: {'last_step': -1, 'due_step': -1, 'due': {}}, 1: {'last_step': -1, 'due_step': -1, 'due': {}}}
_PREEMPT_ENABLED = True
_PREEMPT_FRACTION = 2.0
_PREEMPT_MAX_BATCH = 30
_PREEMPT_MAX_CLONE_DISTANCE = 6
_PREEMPT_MIN_PRICE_RATIO = 0.0
_PREEMPT_MIN_FUTURE_QUANTITY = 4
_PREEMPT_START = 120
_PREEMPT_STOP = 680
_PREMIUM = ('STRAWBERRY', 'MELON', 'MILK', 'WOOL')

def _get(value, key, default = None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, 'get', None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)

def _copy_action(action):
    action = copy.deepcopy(action or {})
    return {'farmer': list(action.get('farmer') or ['PASS']), 'hands': [list(order or ['PASS']) for order in action.get('hands') or []], 'market': [list(order) for order in action.get('market') or []]}

def _seat(obs):
    return 1 if int(_get(obs, 'player', 0) or 0) == 1 else 0

def _farm(obs, seat):
    farms = list(_get(obs, 'farms', []) or [])
    return farms[seat] if seat < len(farms) else {}

def _align_hands(action, obs):
    action = _copy_action(action)
    expected = len(_get(_farm(obs, _seat(obs)), 'hands', []) or [])
    hands = list(action.get('hands') or [])
    if len(hands) < expected:
        hands.extend([['PASS'] for _ in range(expected - len(hands))])
    action['hands'] = [list(order or ['PASS']) for order in hands[:expected]]
    return action

def _shed_access(size):
    half = size // 2
    return {(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)}

def _projected_shed(obs, action):
    farm = _farm(obs, _seat(obs))
    private = _get(obs, 'private', {}) or {}
    projected = {key: max(0, int(value or 0)) for key, value in dict(_get(private, 'shed', {}) or {}).items()}
    inventories = list(_get(private, 'inventories', []) or [])
    positions = [_get(farm, 'farmer', [0, 0]), *list(_get(farm, 'hands', []) or [])]
    unit_actions = [action.get('farmer', ['PASS']), *list(action.get('hands') or [])]
    tiles = list(_get(farm, 'tiles', []) or [])
    access = _shed_access(len(tiles) or 10)
    for index, unit_action in enumerate(unit_actions):
        if index >= len(positions) or index >= len(inventories):
            continue
        position = positions[index]
        if not isinstance(position, (list, tuple)) or len(position) < 2:
            continue
        x, y = (int(position[0]), int(position[1]))
        if (x, y) not in access or not (0 <= y < len(tiles) and 0 <= x < len(tiles[y])):
            continue
        inventory = {key: max(0, int(value or 0)) for key, value in dict(inventories[index] or {}).items()}
        if unit_action and unit_action[0] == 'DROP':
            deposits = inventory.items()
        elif unit_action and unit_action[0] == 'PLACE' and (len(unit_action) >= 2):
            item = unit_action[1]
            tile = tiles[y][x]
            structure = {'COW': 'PASTURE', 'SHEEP': 'PASTURE', 'GOOSE': 'COOP'}.get(item)
            if structure and isinstance(tile, dict) and (tile.get('kind') == structure) and (not tile.get('animal')):
                continue
            try:
                requested = int(unit_action[2]) if len(unit_action) >= 3 else 1
            except (TypeError, ValueError):
                continue
            deposits = ((item, min(max(0, requested), inventory.get(item, 0))),)
        else:
            continue
        for item, quantity in deposits:
            room = max(0, 100 - sum(projected.values()))
            amount = min(max(0, int(quantity or 0)), room)
            if amount:
                projected[item] = projected.get(item, 0) + amount
    return projected

def _public_signature(farm):
    keys = ('WHEAT', 'CARROT', 'TOMATO', 'STRAWBERRY', 'MELON', 'COW', 'SHEEP', 'GOOSE', 'PASTURE', 'COOP', 'WEED')
    counts = {key: 0 for key in keys}
    for row in _get(farm, 'tiles', []) or []:
        for tile in row if isinstance(row, list) else [row]:
            if not isinstance(tile, dict):
                continue
            for field in ('crop', 'animal', 'kind'):
                value = str(tile.get(field, '')).upper()
                if value in counts:
                    counts[value] += 1
                    break
    return (len(_get(farm, 'hands', []) or []), len(_get(farm, 'unlocked_quadrants', []) or []), tuple((counts[key] for key in sorted(counts))))

def _clone_distance(obs):
    farms = list(_get(obs, 'farms', []) or [])
    if len(farms) < 2:
        return 10 ** 9
    left, right = (_public_signature(farms[0]), _public_signature(farms[1]))
    return abs(left[0] - right[0]) + 3 * abs(left[1] - right[1]) + sum((abs(a - b) for a, b in zip(left[2], right[2])))

def _shift_state(obs, step):
    seat = _seat(obs)
    state = _SHIFT_STATE[seat]
    if step == 0 or step < int(state.get('last_step', -1)):
        state = {'last_step': step, 'due_step': -1, 'due': {}}
        _SHIFT_STATE[seat] = state
    state['last_step'] = step
    return state

def _repay_shift(obs, action, step):
    if not _PREEMPT_ENABLED:
        return action
    state = _shift_state(obs, step)
    if int(state.get('due_step', -1)) != step:
        if int(state.get('due_step', -1)) < step:
            state['due_step'], state['due'] = (-1, {})
        return action
    due = {item: max(0, int(quantity)) for item, quantity in dict(state.get('due') or {}).items()}
    market = []
    for raw in action.get('market', []) or []:
        order = list(raw)
        if len(order) >= 3 and order[0] == 'SELL' and (due.get(order[1], 0) > 0):
            item = order[1]
            requested = max(0, int(order[2]))
            reduction = min(requested, due[item])
            requested -= reduction
            due[item] -= reduction
            if requested <= 0:
                continue
            order[2] = requested
        market.append(order)
    action['market'] = market
    state['due_step'], state['due'] = (-1, {})
    return action

def _future_sells_at(step, horizon):
    if step + horizon >= len(_ACTIONS):
        return {}
    result = {}
    for raw in _ACTIONS[step + horizon].get('market') or []:
        if len(raw) >= 3 and raw[0] == 'SELL' and (raw[1] in _PREMIUM):
            result[raw[1]] = result.get(raw[1], 0) + max(0, int(raw[2]))
    return result

def _preempt_shift(obs, action, step):
    if not _PREEMPT_ENABLED or not _PREEMPT_START <= step < _PREEMPT_STOP:
        return action
    state = _shift_state(obs, step)
    if state.get('due') or _clone_distance(obs) > _PREEMPT_MAX_CLONE_DISTANCE:
        return action
    market = list(action.get('market') or [])
    if len(market) >= 10:
        return action
    remaining = _projected_shed(obs, action)
    for raw in market:
        if len(raw) >= 3 and raw[0] == 'SELL':
            item = raw[1]
            remaining[item] = max(0, int(remaining.get(item, 0) or 0) - max(0, int(raw[2])))
    prices = _get(_get(obs, 'market', {}) or {}, 'prices', {}) or {}
    for horizon in (3, 2, 1):
        future = _future_sells_at(step, horizon)
        if not future:
            continue
        shifted = {}
        trial_market = list(market)
        trial_remaining = dict(remaining)
        for item in _PREMIUM:
            future_quantity = max(0, int(future.get(item, 0) or 0))
            if future_quantity < _PREEMPT_MIN_FUTURE_QUANTITY:
                continue
            base_price = float(_MARKET_PARAMS[item][0])
            current_price = float(_get(prices, item, 0) or 0)
            if current_price <= _PRICE_FLOOR:
                continue
            if current_price < base_price * _PREEMPT_MIN_PRICE_RATIO:
                continue
            target = min(max(0, int(trial_remaining.get(item, 0) or 0)), future_quantity, _PREEMPT_MAX_BATCH, max(1, int(round(future_quantity * _PREEMPT_FRACTION))))
            if target <= 0 or len(trial_market) >= 10:
                continue
            trial_market.append(['SELL', item, target])
            trial_remaining[item] = max(0, int(trial_remaining.get(item, 0) or 0) - target)
            shifted[item] = target
        if shifted:
            action['market'] = trial_market[:10]
            state['due_step'] = step + horizon
            state['due'] = shifted
            return action
    return action

def _tile_at(farm, position):
    try:
        x, y = (int(position[0]), int(position[1]))
        return (_get(farm, 'tiles', []) or [])[y][x]
    except (IndexError, TypeError, ValueError):
        return 'LOCKED'

def _trace_actor_action(step, actor):
    trace = _ACTIONS[min(max(int(step), 0), len(_ACTIONS) - 1)] or {}
    if actor == 'farmer':
        return list(trace.get('farmer') or ['PASS'])
    hands = trace.get('hands', []) or []
    return list(hands[actor] if actor < len(hands) else ['PASS'])

def _weed_repair_action(obs, action, step):
    action = _align_hands(action, obs)
    seat = _seat(obs)
    game = _WEED_STATE[seat]
    if step == 0 or step < game.get('last_step', -1):
        game = {'last_step': step, 'active': {}}
        _WEED_STATE[seat] = game
    game['last_step'] = step
    farm = _farm(obs, seat)
    positions = [_get(farm, 'farmer'), *list(_get(farm, 'hands', []) or [])]
    unit_actions = [action.get('farmer', ['PASS']), *list(action.get('hands') or [])]
    active = game['active']
    for actor, transaction in list(active.items()):
        index = 0 if actor == 'farmer' else int(actor) + 1
        if index >= len(unit_actions):
            active.pop(actor, None)
            continue
        age = step - transaction['start']
        if age == 1:
            unit_actions[index] = list(transaction['intended'])
        elif 2 <= age <= 1 + _WEED_REPLAY_STEPS:
            unit_actions[index] = _trace_actor_action(step - 1, actor)
        else:
            active.pop(actor, None)
    for index, (position, intended) in enumerate(zip(positions, unit_actions)):
        actor = 'farmer' if index == 0 else index - 1
        if actor in active or not isinstance(intended, list) or (not intended):
            continue
        if intended[0] not in ('BUILD_PASTURE', 'PLANT'):
            continue
        tile = _tile_at(farm, position)
        if not isinstance(tile, dict) or tile.get('kind') != 'WEED':
            continue
        active[actor] = {'start': step, 'intended': list(intended)}
        unit_actions[index] = ['DIG']
    action['farmer'] = unit_actions[0] if unit_actions else ['PASS']
    action['hands'] = unit_actions[1:]
    return _align_hands(action, obs)

def _shape(name, value):
    value = max(0.0, float(value))
    if name == 'linear':
        return value
    if name == 'sq':
        return value * value
    if name == 'sqrt':
        return math.sqrt(value)
    if name == 'log':
        return math.log1p(value)
    if name == 'log10':
        return math.log10(1.0 + value)
    raise ValueError(name)

def _market_price(item, inventory):
    base, equilibrium, scale, below_func, below_target, above_func, above_target = _MARKET_PARAMS[item]
    if inventory < equilibrium:
        amplitude = below_target * base / _shape(below_func, scale)
        price = base + amplitude * _shape(below_func, equilibrium - inventory)
    else:
        amplitude = above_target * base / _shape(above_func, scale)
        price = base - amplitude * _shape(above_func, inventory - equilibrium)
    return max(_PRICE_FLOOR, int(round(price)))

def _is_sell(order):
    return isinstance(order, (list, tuple)) and len(order) >= 3 and (order[0] == 'SELL') and (order[1] in _MARKET_PARAMS)

def _impact_score(obs, order):
    if not _is_sell(order):
        return float('-inf')
    item = str(order[1])
    try:
        quantity = max(0, int(order[2]))
    except (TypeError, ValueError):
        return 0.0
    market = _get(obs, 'market', {}) or {}
    inventory = _get(market, 'inventory', {}) or {}
    prices = _get(market, 'prices', {}) or {}
    current_inventory = int(_get(inventory, item, 10000) or 0)
    current_quote = float(_get(prices, item, _market_price(item, current_inventory)) or 0)
    later_quote = float(_market_price(item, current_inventory + quantity))
    return float(quantity) * max(0.0, current_quote - later_quote)

def _demand_per_day(obs, configuration, item):
    town = _get(obs, 'town', {}) or {}
    shops = list(_get(town, 'unlocked_shops', []) or [])
    turns_per_day = int(_get(configuration, 'turnsPerDay', 24) or 24)
    shop_interval = max(1, int(_get(configuration, 'townShopSellInterval', 4) or 4))
    demand = 0.0
    for shop in shops:
        products = _SHOP_PRODUCTS.get(shop, ())
        if item in products:
            demand += turns_per_day / shop_interval * (2 if len(products) == 1 else 1)
    if item != 'FERTILIZER':
        center_interval = max(1, int(_get(configuration, 'townCenterSellInterval', 24) or 24))
        demand += turns_per_day / center_interval
    return demand

def _order_score(obs, configuration, order):
    score = _impact_score(obs, order)
    if score <= 0 or not _is_sell(order):
        return score
    item = str(order[1])
    quantity = max(0, int(order[2]))
    market = _get(obs, 'market', {}) or {}
    inventory = _get(market, 'inventory', {}) or {}
    current_inventory = int(_get(inventory, item, 10000) or 0)
    demand = max(0.25, _demand_per_day(obs, configuration, item))
    excess = max(0.0, current_inventory + quantity - 10000)
    urgency = min(1.0, excess / demand / 10.0)
    return score * (1.0 + _DEMAND_ALPHA * urgency)

def _rank_sell_slots(obs, action, configuration):
    action = _copy_action(action)
    market = list(action.get('market') or [])
    rows = [(_order_score(obs, configuration, order), -index, list(order)) for index, order in enumerate(market) if _is_sell(order)]
    if len(rows) < 2:
        return action
    rows.sort(reverse = True)
    ranked = iter((row[2] for row in rows))
    action['market'] = [next(ranked) if _is_sell(order) else order for order in market]
    return action

def _terminal_liquidation(obs, action, step):
    if step < 716:
        return action
    action = _copy_action(action)
    shed = _get(_get(obs, 'private', {}) or {}, 'shed', {}) or {}
    planned = {item: 0 for item in _SELLABLE}
    for order in action.get('market', []):
        if _is_sell(order):
            planned[str(order[1])] += max(0, int(order[2]))
    for item in _LIQUIDATION_ORDER:
        available = max(0, int(_get(shed, item, 0) or 0))
        extra = available if step >= 718 else max(0, available - planned[item])
        if extra and len(action['market']) < 10:
            action['market'].append(['SELL', item, extra])
    return action

def agent(obs):
    try:
        step = min(max(0, int(_get(obs, 'step', 0) or 0)), len(_ACTIONS) - 1)
        action = _weed_repair_action(obs, _copy_action(_ACTIONS[step]), step)
        action = _repay_shift(obs, action, step)
        action = _rank_sell_slots(obs, action, None)
        action = _preempt_shift(obs, action, step)
        action = _terminal_liquidation(obs, action, step)
        return _align_hands(action, obs)
    except Exception:
        farm = _farm(obs, _seat(obs))
        return {'farmer': ['PASS'], 'hands': [['PASS'] for _ in _get(farm, 'hands', []) or []], 'market': []}

def _kaggle_submission_entrypoint(obs):
    return agent(obs)

## Notebook display setup

Configures pandas and a small CSS override so wide DataFrames render in full
(no truncated columns or scroll boxes) throughout the rest of the notebook.
Purely cosmetic — no effect on the agent itself.

In [ ]:
import pandas as pd
from IPython.display import HTML

# Configure complete and readable DataFrame output across the notebook
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

table_style = """
<style>
.dataframe { width: 100% !important; table-layout: auto !important; }
.dataframe th { white-space: nowrap !important; text-align: left !important; }
.dataframe td { white-space: normal !important; word-break: normal !important; overflow-wrap: anywhere !important; text-align: left !important; vertical-align: top !important; }
.output_scroll { height: auto !important; max-height: none !important; }
.output_subarea { max-width: 100% !important; overflow-x: visible !important; }
</style>
"""

display(HTML(table_style))

## Decoding the embedded route for inspection

Loads `main.py` as a Python module without modifying it, and decodes the
embedded `_ACTIONS` route back into a plain list of per-step actions. Each
step is unpacked into `route_df` (day, hour, farmer/hand/market actions), and
three counters (`farmer_counter`, `hand_counter`, `market_counter`) tally how
often each action type appears. This is the foundation every later analysis
cell builds on.

In [ ]:
import hashlib, importlib.util, json
from collections import Counter
from pathlib import Path

import pandas as pd

# Inspect the generated agent and decode the embedded strategy without modifying main.py
main_path = Path("/kaggle/working/main.py")

if not main_path.exists():
    raise FileNotFoundError(f"main.py not found: {main_path}")

main_bytes_before_analysis = main_path.read_bytes()
main_sha256_before_analysis = hashlib.sha256(main_bytes_before_analysis).hexdigest()
module_spec = importlib.util.spec_from_file_location("adaptive_agent_analysis", main_path)

if module_spec is None or module_spec.loader is None:
    raise RuntimeError("Unable to import main.py for analysis.")

agent_module = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(agent_module)
actions = getattr(agent_module, "_ACTIONS", None)

if not isinstance(actions, list) or not actions:
    raise RuntimeError("Embedded _ACTIONS route could not be decoded.")

route_records = []
farmer_counter = Counter()
hand_counter = Counter()
market_counter = Counter()
full_action_counter = Counter()

for step, action in enumerate(actions):
    if not isinstance(action, dict):
        continue

    day = step // 24
    hour = step % 24
    farmer = action.get("farmer", ["PASS"])
    hands = action.get("hands", [])
    market = action.get("market", [])
    farmer_name = str(farmer[0]) if isinstance(farmer, list) and farmer else "PASS"
    farmer_counter[farmer_name] += 1
    hand_non_pass = 0

    for hand_action in hands:
        hand_name = str(hand_action[0]) if isinstance(hand_action, list) and hand_action else "PASS"
        hand_counter[hand_name] += 1
        hand_non_pass += int(hand_name != "PASS")

    market_count = len(market) if isinstance(market, list) else 0

    if isinstance(market, list):
        for market_action in market:
            market_name = str(market_action[0]) if isinstance(market_action, list) and market_action else "UNKNOWN"
            market_counter[market_name] += 1

    farmer_non_pass = int(farmer_name != "PASS")
    action_key = json.dumps(action, sort_keys = True, separators = (",", ":"))
    full_action_counter[action_key] += 1
    route_records.append({"step": step, "day": day, "hour": hour, "farmer_action": farmer_name, "farmer_non_pass": farmer_non_pass, "hand_count": len(hands) if isinstance(hands, list) else 0, "hand_non_pass": hand_non_pass, "market_actions": market_count, "total_active_actions": farmer_non_pass + hand_non_pass + market_count})

route_df = pd.DataFrame(route_records)

if route_df.empty:
    raise RuntimeError("No valid strategy records were created.")

total_steps = len(route_df)
distinct_full_actions = len(full_action_counter)
farmer_non_pass_total = int(route_df["farmer_non_pass"].sum())
hand_non_pass_total = int(route_df["hand_non_pass"].sum())
market_action_total = int(route_df["market_actions"].sum())
total_activity = int(route_df["total_active_actions"].sum())
active_steps = int((route_df["total_active_actions"] > 0).sum())
passive_steps = total_steps - active_steps
activity_rate = active_steps / total_steps if total_steps else 0.0

strategy_inspection_df = pd.DataFrame({"Metric": ["Embedded Strategy Steps", "Observed Days", "Observed Hours per Day", "Distinct Full Action Templates", "Farmer Non Pass Actions", "Hands Non Pass Actions", "Market Actions", "Total Active Actions", "Active Strategy Steps", "Passive Strategy Steps", "Activity Rate", "Analysis Mode"], "Value": [f"{total_steps:,}", f"{route_df['day'].nunique():,}", f"{route_df['hour'].nunique():,}", f"{distinct_full_actions:,}", f"{farmer_non_pass_total:,}", f"{hand_non_pass_total:,}", f"{market_action_total:,}", f"{total_activity:,}", f"{active_steps:,}", f"{passive_steps:,}", f"{activity_rate:.2%}", "READ ONLY"]})
display(strategy_inspection_df)

## Full action vocabulary across all actors

Combines the farmer, hand, and market counters into one long table
(`action_vocabulary_df`) listing every distinct action name, which actor
issues it, and how often — then aggregates that into `combined_action_df`,
a single ranked count per action type regardless of actor.

In [ ]:
from collections import Counter
import pandas as pd

# Analyze the complete action vocabulary across farmer hands and market actors
required_objects = ["actions", "route_df", "farmer_counter", "hand_counter", "market_counter"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required strategy inspection objects are missing: " + ", ".join(missing_objects))

actor_action_rows = []

for action_name, count in farmer_counter.items():
    actor_action_rows.append({"Actor": "Farmer", "Action": str(action_name), "Count": int(count)})

for action_name, count in hand_counter.items():
    actor_action_rows.append({"Actor": "Hands", "Action": str(action_name), "Count": int(count)})

for action_name, count in market_counter.items():
    actor_action_rows.append({"Actor": "Market", "Action": str(action_name), "Count": int(count)})

action_vocabulary_df = pd.DataFrame(actor_action_rows)

if action_vocabulary_df.empty:
    raise RuntimeError("Action vocabulary analysis produced no records.")

action_vocabulary_df = action_vocabulary_df.sort_values(["Count", "Actor", "Action"], ascending = [False, True, True]).reset_index(drop = True)

combined_action_counter = Counter()

for row in action_vocabulary_df.itertuples(index = False):
    combined_action_counter[str(row.Action)] += int(row.Count)

combined_action_df = pd.DataFrame([{"Action": action, "Count": int(count)} for action, count in combined_action_counter.items()])
combined_action_df = combined_action_df.sort_values(["Count", "Action"], ascending = [False, True]).reset_index(drop = True)
combined_action_df["Share"] = combined_action_df["Count"] / combined_action_df["Count"].sum()
combined_action_df.insert(0, "Rank", range(1, len(combined_action_df) + 1))

non_pass_action_df = combined_action_df[combined_action_df["Action"] != "PASS"].copy().reset_index(drop = True)

actor_totals_df = action_vocabulary_df.groupby("Actor", as_index = False)["Count"].sum()
actor_totals_df = actor_totals_df.sort_values("Count", ascending = False).reset_index(drop = True)
actor_totals_df["Share"] = actor_totals_df["Count"] / actor_totals_df["Count"].sum()

unique_farmer_actions = int(action_vocabulary_df.loc[action_vocabulary_df["Actor"] == "Farmer", "Action"].nunique())
unique_hand_actions = int(action_vocabulary_df.loc[action_vocabulary_df["Actor"] == "Hands", "Action"].nunique())
unique_market_actions = int(action_vocabulary_df.loc[action_vocabulary_df["Actor"] == "Market", "Action"].nunique())
unique_combined_actions = int(combined_action_df["Action"].nunique())
total_recorded_actions = int(combined_action_df["Count"].sum())
pass_count = int(combined_action_counter.get("PASS", 0))
non_pass_count = total_recorded_actions - pass_count
pass_rate = pass_count / total_recorded_actions if total_recorded_actions else 0.0
non_pass_rate = non_pass_count / total_recorded_actions if total_recorded_actions else 0.0

action_vocabulary_summary_df = pd.DataFrame({"Metric": ["Unique Farmer Actions", "Unique Hand Actions", "Unique Market Actions", "Unique Combined Actions", "Total Recorded Actions", "PASS Actions", "Non PASS Actions", "PASS Rate", "Non PASS Rate"], "Value": [f"{unique_farmer_actions:,}", f"{unique_hand_actions:,}", f"{unique_market_actions:,}", f"{unique_combined_actions:,}", f"{total_recorded_actions:,}", f"{pass_count:,}", f"{non_pass_count:,}", f"{pass_rate:.2%}", f"{non_pass_rate:.2%}"]})
combined_action_display_df = combined_action_df.head(20).copy()
combined_action_display_df["Share"] = combined_action_display_df["Share"].map(lambda value: f"{value:.2%}")

display(action_vocabulary_summary_df)
display(combined_action_display_df)

## Chart: most frequent actions overall

Horizontal bar chart of the top 20 actions by raw frequency across the whole
route. Gives a quick read on what the embedded strategy spends most of its
turns doing.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Visualize the most frequent actions with distinct colors
required_objects = ["combined_action_df"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required action vocabulary objects are missing: " + ", ".join(missing_objects))

top_action_count = min(20, len(combined_action_df))
top_actions_df = combined_action_df.head(top_action_count).copy()
top_actions_df = top_actions_df.sort_values("Count", ascending = True).reset_index(drop = True)

sns.set_theme(style = "whitegrid")
figure, axis = plt.subplots(figsize = (12, 8))
palette = sns.color_palette("husl", n_colors = len(top_actions_df))
bars = axis.barh(top_actions_df["Action"], top_actions_df["Count"], color = palette)
maximum_count = float(top_actions_df["Count"].max()) if not top_actions_df.empty else 0.0

for bar, count in zip(bars, top_actions_df["Count"]):
    label_offset = maximum_count * 0.01 if maximum_count > 0 else 0.1
    axis.text(bar.get_width() + label_offset, bar.get_y() + bar.get_height() / 2, f"{int(count):,}", va = "center", fontsize = 10)

axis.set_title("Top Actions Across the Embedded Strategy", fontsize = 16, pad = 15)
axis.set_xlabel("Action Frequency", fontsize = 11)
axis.set_ylabel("Action", fontsize = 11)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(axis = "x", alpha = 0.25)
axis.grid(axis = "y", visible = False)

plt.tight_layout()
plt.show()

most_frequent_action = combined_action_df.iloc[0]
top_actions_total = int(combined_action_df.head(top_action_count)["Count"].sum())
all_actions_total = int(combined_action_df["Count"].sum())
top_actions_share = top_actions_total / all_actions_total if all_actions_total else 0.0

action_visualization_summary_df = pd.DataFrame({"Metric": ["Displayed Action Types", "Total Action Types", "Most Frequent Action", "Most Frequent Count", "Most Frequent Share", "Top Actions Combined Share", "Color Scheme", "Main File Modification"], "Value": [f"{top_action_count:,}", f"{len(combined_action_df):,}", str(most_frequent_action["Action"]), f"{int(most_frequent_action['Count']):,}", f"{most_frequent_action['Share']:.2%}", f"{top_actions_share:.2%}", "DISTINCT MULTICOLOR", "NONE"]})
display(action_visualization_summary_df)

## Chart: daily action volume by actor

Line chart of non-PASS actions per day, split by farmer, hands, and market,
plus a combined total. Shows how activity ramps up or shifts in character as
the game progresses (e.g. more hands acting once the workforce grows).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Visualize daily strategy activity across farmer hands and market actors
required_objects = ["route_df"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required route analysis objects are missing: " + ", ".join(missing_objects))

daily_activity_df = route_df.groupby("day", as_index = False).agg(farmer_actions = ("farmer_non_pass", "sum"), hand_actions = ("hand_non_pass", "sum"), market_actions = ("market_actions", "sum"), total_actions = ("total_active_actions", "sum"))

if daily_activity_df.empty:
    raise RuntimeError("Daily strategy activity could not be calculated.")

daily_activity_df["farmer_actions"] = daily_activity_df["farmer_actions"].astype(int)
daily_activity_df["hand_actions"] = daily_activity_df["hand_actions"].astype(int)
daily_activity_df["market_actions"] = daily_activity_df["market_actions"].astype(int)
daily_activity_df["total_actions"] = daily_activity_df["total_actions"].astype(int)

sns.set_theme(style = "whitegrid")
figure, axis = plt.subplots(figsize = (14, 7))

axis.plot(daily_activity_df["day"], daily_activity_df["farmer_actions"], color = "#E63946", marker = "o", linewidth = 2.3, markersize = 5, label = "Farmer")
axis.plot(daily_activity_df["day"], daily_activity_df["hand_actions"], color = "#4361EE", marker = "s", linewidth = 2.3, markersize = 5, label = "Hands")
axis.plot(daily_activity_df["day"], daily_activity_df["market_actions"], color = "#F4A261", marker = "^", linewidth = 2.3, markersize = 5, label = "Market")
axis.plot(daily_activity_df["day"], daily_activity_df["total_actions"], color = "#6C757D", linestyle = "--", linewidth = 1.8, alpha = 0.75, label = "Total")
axis.fill_between(daily_activity_df["day"], daily_activity_df["total_actions"], color = "#ADB5BD", alpha = 0.08)

axis.set_title("Daily Activity Across the Embedded 720 Step Strategy", fontsize = 16, pad = 15)
axis.set_xlabel("Game Day", fontsize = 11)
axis.set_ylabel("Number of Active Actions", fontsize = 11)
axis.set_xticks(daily_activity_df["day"])
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(alpha = 0.22)
axis.legend(frameon = False, ncol = 4, loc = "upper center")

plt.tight_layout()
plt.show()

peak_total_row = daily_activity_df.loc[daily_activity_df["total_actions"].idxmax()]
peak_farmer_row = daily_activity_df.loc[daily_activity_df["farmer_actions"].idxmax()]
peak_hand_row = daily_activity_df.loc[daily_activity_df["hand_actions"].idxmax()]
peak_market_row = daily_activity_df.loc[daily_activity_df["market_actions"].idxmax()]

daily_activity_summary_df = pd.DataFrame({"Metric": ["Observed Days", "Total Farmer Actions", "Total Hand Actions", "Total Market Actions", "Total Active Actions", "Peak Overall Activity Day", "Peak Overall Actions", "Peak Farmer Activity Day", "Peak Farmer Actions", "Peak Hand Activity Day", "Peak Hand Actions", "Peak Market Activity Day", "Peak Market Actions", "Main File Modification"], "Value": [f"{len(daily_activity_df):,}", f"{int(daily_activity_df['farmer_actions'].sum()):,}", f"{int(daily_activity_df['hand_actions'].sum()):,}", f"{int(daily_activity_df['market_actions'].sum()):,}", f"{int(daily_activity_df['total_actions'].sum()):,}", f"Day {int(peak_total_row['day'])}", f"{int(peak_total_row['total_actions']):,}", f"Day {int(peak_farmer_row['day'])}", f"{int(peak_farmer_row['farmer_actions']):,}", f"Day {int(peak_hand_row['day'])}", f"{int(peak_hand_row['hand_actions']):,}", f"Day {int(peak_market_row['day'])}", f"{int(peak_market_row['market_actions']):,}", "NONE"]})
display(daily_activity_summary_df)

## Heatmap: activity by day and hour

Day-by-hour heatmap of total active (non-PASS) actions. Surfaces intra-day
timing patterns — for example, whether activity clusters at the start of
each day (hiring, planting) versus spreading evenly across all 24 hours.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Visualize strategy intensity across every game day and hour
required_objects = ["route_df"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required route analysis objects are missing: " + ", ".join(missing_objects))

activity_heatmap_df = route_df.pivot_table(index = "day", columns = "hour", values = "total_active_actions", aggfunc = "sum", fill_value = 0)
activity_heatmap_df = activity_heatmap_df.sort_index(axis = 0).sort_index(axis = 1)

if activity_heatmap_df.empty:
    raise RuntimeError("Strategy activity heatmap could not be generated.")

sns.set_theme(style = "white")
figure, axis = plt.subplots(figsize = (16, 10))

sns.heatmap(activity_heatmap_df, cmap = "turbo", linewidths = 0.25, linecolor = "white", annot = True, fmt = ".0f", annot_kws = {"fontsize": 7}, cbar_kws = {"label": "Active Actions"}, ax = axis)

axis.set_title("Strategy Activity Heatmap by Day and Hour", fontsize = 16, pad = 15)
axis.set_xlabel("Game Hour", fontsize = 11)
axis.set_ylabel("Game Day", fontsize = 11)
axis.tick_params(axis = "x", rotation = 0)
axis.tick_params(axis = "y", rotation = 0)

plt.tight_layout()
plt.show()

maximum_activity = int(activity_heatmap_df.to_numpy().max())
minimum_activity = int(activity_heatmap_df.to_numpy().min())
average_activity = float(activity_heatmap_df.to_numpy().mean())
median_activity = float(pd.Series(activity_heatmap_df.to_numpy().ravel()).median())
active_cells = int((activity_heatmap_df.to_numpy() > 0).sum())
total_cells = int(activity_heatmap_df.size)
active_cell_rate = active_cells / total_cells if total_cells else 0.0

maximum_locations = []

for day in activity_heatmap_df.index:
    for hour in activity_heatmap_df.columns:
        if int(activity_heatmap_df.loc[day, hour]) == maximum_activity:
            maximum_locations.append({"Day": int(day), "Hour": int(hour), "Active Actions": maximum_activity})

heatmap_summary_df = pd.DataFrame({"Metric": ["Heatmap Days", "Heatmap Hours", "Heatmap Cells", "Active Cells", "Active Cell Rate", "Minimum Activity per Step", "Maximum Activity per Step", "Average Activity per Step", "Median Activity per Step", "Peak Activity Locations", "Color Scale", "Main File Modification"], "Value": [f"{activity_heatmap_df.shape[0]:,}", f"{activity_heatmap_df.shape[1]:,}", f"{total_cells:,}", f"{active_cells:,}", f"{active_cell_rate:.2%}", f"{minimum_activity:,}", f"{maximum_activity:,}", f"{average_activity:.2f}", f"{median_activity:.2f}", f"{len(maximum_locations):,}", "TURBO MULTICOLOR", "NONE"]})
peak_activity_df = pd.DataFrame(maximum_locations)
display(heatmap_summary_df)

if not peak_activity_df.empty:
    display(peak_activity_df.head(10))

## Market action breakdown

Extracts every market-side action (`BUY_SEED`, `BUY_PRODUCT`, `SELL`, `HIRE`,
`BUY_ANIMAL`, `BUY_LAND`, ...) from the route into `market_strategy_df`, then
counts how often each market action type is used across the full 720-step
route.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Analyze and visualize market behavior across the complete strategy
required_objects = ["actions", "route_df"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required embedded strategy objects are missing: " + ", ".join(missing_objects))

market_records = []

for step, action in enumerate(actions):
    if not isinstance(action, dict):
        continue

    day = step // 24
    hour = step % 24
    market_actions = action.get("market", [])

    if not isinstance(market_actions, list):
        continue

    for market_action in market_actions:
        if not isinstance(market_action, list) or not market_action:
            continue

        action_name = str(market_action[0])
        argument_one = str(market_action[1]) if len(market_action) > 1 else ""
        argument_two = str(market_action[2]) if len(market_action) > 2 else ""
        market_records.append({"step": step, "day": day, "hour": hour, "action": action_name, "arg1": argument_one, "arg2": argument_two})

market_strategy_df = pd.DataFrame(market_records)

if market_strategy_df.empty:
    raise RuntimeError("No market actions were found in the embedded strategy.")

market_action_counts_df = market_strategy_df.groupby("action", as_index = False).size()
market_action_counts_df = market_action_counts_df.rename(columns = {"size": "count"})
market_action_counts_df = market_action_counts_df.sort_values(["count", "action"], ascending = [False, True]).reset_index(drop = True)
market_action_counts_df["share"] = market_action_counts_df["count"] / market_action_counts_df["count"].sum()
market_action_counts_df.insert(0, "rank", range(1, len(market_action_counts_df) + 1))

market_daily_df = market_strategy_df.groupby(["day", "action"]).size().unstack(fill_value = 0)
market_daily_df = market_daily_df.reindex(range(int(route_df["day"].min()), int(route_df["day"].max()) + 1), fill_value = 0)

market_action_names = market_action_counts_df["action"].tolist()
market_daily_df = market_daily_df.reindex(columns = market_action_names, fill_value = 0)

sns.set_theme(style = "whitegrid")
figure, axis = plt.subplots(figsize = (15, 8))
palette = sns.color_palette("husl", n_colors = len(market_action_names))
bottom_values = pd.Series(0, index = market_daily_df.index, dtype = float)

for action_name, color in zip(market_action_names, palette):
    values = market_daily_df[action_name].astype(float)
    axis.bar(market_daily_df.index, values, bottom = bottom_values, label = action_name, color = color, width = 0.82)
    bottom_values = bottom_values + values

axis.set_title("Daily Market Strategy Breakdown", fontsize = 16, pad = 15)
axis.set_xlabel("Game Day", fontsize = 11)
axis.set_ylabel("Number of Market Actions", fontsize = 11)
axis.set_xticks(market_daily_df.index)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(axis = "y", alpha = 0.22)
axis.grid(axis = "x", visible = False)
axis.legend(title = "Market Action", frameon = False, bbox_to_anchor = (1.02, 1), loc = "upper left")

plt.tight_layout()
plt.show()

market_daily_totals = market_daily_df.sum(axis = 1)
peak_market_day = int(market_daily_totals.idxmax())
peak_market_actions = int(market_daily_totals.max())
total_market_actions = int(len(market_strategy_df))
unique_market_actions = int(market_strategy_df["action"].nunique())
active_market_days = int((market_daily_totals > 0).sum())
inactive_market_days = int((market_daily_totals == 0).sum())

market_strategy_summary_df = pd.DataFrame({"Metric": ["Total Market Actions", "Unique Market Action Types", "Active Market Days", "Inactive Market Days", "Peak Market Day", "Peak Market Day Actions", "Main File Modification"], "Value": [f"{total_market_actions:,}", f"{unique_market_actions:,}", f"{active_market_days:,}", f"{inactive_market_days:,}", f"Day {peak_market_day}", f"{peak_market_actions:,}", "NONE"]})

market_action_display_df = market_action_counts_df.copy()
market_action_display_df.columns = ["Rank", "Action", "Count", "Share"]
market_action_display_df["Share"] = market_action_display_df["Share"].map(lambda value: f"{value:.2%}")

display(market_strategy_summary_df)
display(market_action_display_df)

## Product-level market behavior

Cross-tabulates market actions against the specific product they target
(e.g. how many `SELL` vs `BUY_PRODUCT` orders touch wheat, melon, wool, ...),
rendered as a heatmap ranked by total trade volume per product.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Analyze product level market behavior across the embedded strategy
required_objects = ["market_strategy_df"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required market strategy objects are missing: " + ", ".join(missing_objects))

product_market_df = market_strategy_df.copy()
product_market_df["item"] = product_market_df["arg1"].astype(str).str.strip()
product_market_df["item"] = product_market_df["item"].replace({"": "NO_ITEM", "None": "NO_ITEM", "nan": "NO_ITEM"})

market_product_counts_df = product_market_df.groupby(["item", "action"]).size().unstack(fill_value = 0)

if market_product_counts_df.empty:
    raise RuntimeError("Market product action matrix could not be generated.")

market_product_counts_df["total"] = market_product_counts_df.sum(axis = 1)
market_product_counts_df = market_product_counts_df.sort_values("total", ascending = False)

maximum_products = min(20, len(market_product_counts_df))
market_product_heatmap_df = market_product_counts_df.head(maximum_products).drop(columns = "total")
market_product_summary_df = market_product_counts_df.reset_index()
market_product_summary_df.insert(0, "rank", range(1, len(market_product_summary_df) + 1))

sns.set_theme(style = "white")
figure, axis = plt.subplots(figsize = (13, max(6, maximum_products * 0.55)))

sns.heatmap(market_product_heatmap_df, cmap = "magma", linewidths = 0.5, linecolor = "white", annot = True, fmt = ".0f", annot_kws = {"fontsize": 9}, cbar_kws = {"label": "Action Count"}, ax = axis)

axis.set_title("Market Product and Action Matrix", fontsize = 16, pad = 15)
axis.set_xlabel("Market Action", fontsize = 11)
axis.set_ylabel("Product or Resource", fontsize = 11)
axis.tick_params(axis = "x", rotation = 35)
axis.tick_params(axis = "y", rotation = 0)

plt.tight_layout()
plt.show()

product_totals = market_product_counts_df["total"].sort_values(ascending = False)
most_active_product = str(product_totals.index[0])
most_active_product_count = int(product_totals.iloc[0])
unique_products = int(len(market_product_counts_df))
unique_market_action_types = int(len(market_product_heatmap_df.columns))
matrix_total_actions = int(market_product_heatmap_df.to_numpy().sum())

market_product_analysis_summary_df = pd.DataFrame({"Metric": ["Unique Products or Resources", "Market Action Types in Matrix", "Displayed Products", "Displayed Matrix Actions", "Most Active Product or Resource", "Most Active Product Action Count", "Color Scale", "Main File Modification"], "Value": [f"{unique_products:,}", f"{unique_market_action_types:,}", f"{maximum_products:,}", f"{matrix_total_actions:,}", most_active_product, f"{most_active_product_count:,}", "MAGMA MULTICOLOR", "NONE"]})

market_product_display_df = market_product_summary_df.head(15).copy()
market_product_display_df = market_product_display_df.rename(columns = {"rank": "Rank", "item": "Product or Resource", "total": "Total Actions"})

display(market_product_analysis_summary_df)
display(market_product_display_df)

## Locating the real environment source for cross-validation

Rather than trusting only the competition's written rubric, this cell finds
the actual installed Kaggriculture environment source inside the
`kaggle_environments` package and searches its files for the real pricing,
market, and inventory implementation — the ground truth the agent's pricing
assumptions get checked against next.

In [ ]:
import contextlib, hashlib, io, logging, re
from pathlib import Path

import pandas as pd

# Discover the original Kaggriculture market pricing implementation without noisy import logs
logging.disable(logging.CRITICAL)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import kaggle_environments

logging.disable(logging.NOTSET)

required_objects = ["main_path", "main_sha256_before_analysis"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required notebook objects are missing: " + ", ".join(missing_objects))

environment_package_path = Path(kaggle_environments.__file__).resolve().parent / "envs" / "kaggriculture"

if not environment_package_path.exists():
    raise RuntimeError(f"Kaggriculture environment source directory was not found: {environment_package_path}")

python_files = sorted(environment_package_path.rglob("*.py"))

if not python_files:
    raise RuntimeError("No Python source files were found inside the Kaggriculture environment.")

search_terms = ["price", "prices", "inventory", "market", "supply", "demand", "wheat", "carrot", "strawberry", "tomato", "melon", "milk", "egg", "wool", "fertilizer"]
pricing_candidates = []

for source_path in python_files:
    try:
        source_text = source_path.read_text(encoding = "utf-8")
    except Exception:
        continue

    source_lines = source_text.splitlines()

    for line_number, line in enumerate(source_lines, start = 1):
        lower_line = line.lower()
        matched_terms = sorted({term for term in search_terms if term in lower_line})

        if not matched_terms:
            continue

        pricing_candidates.append({"File": str(source_path.relative_to(environment_package_path)), "Line": int(line_number), "Matched Terms": ", ".join(matched_terms), "Source": line.strip()})

pricing_source_df = pd.DataFrame(pricing_candidates)

if pricing_source_df.empty:
    raise RuntimeError("No pricing related source code could be discovered.")

pricing_priority_pattern = re.compile(r"price|pricing|market.*inventory|inventory.*price|supply|demand", re.IGNORECASE)
pricing_source_df["Priority"] = pricing_source_df["Source"].apply(lambda value: int(bool(pricing_priority_pattern.search(str(value)))))
pricing_source_df = pricing_source_df.sort_values(["Priority", "File", "Line"], ascending = [False, True, True]).reset_index(drop = True)

function_pattern = re.compile(r"^\s*def\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(")
function_candidates = []

for source_path in python_files:
    try:
        source_text = source_path.read_text(encoding = "utf-8")
    except Exception:
        continue

    source_lines = source_text.splitlines()
    current_function = None
    current_start = None
    current_lines = []

    for line_number, line in enumerate(source_lines, start = 1):
        function_match = function_pattern.match(line)

        if function_match:
            if current_function is not None:
                function_source = "\n".join(current_lines)
                lower_function_source = function_source.lower()
                matched_terms = sorted({term for term in search_terms if term in lower_function_source})

                if "price" in lower_function_source or "prices" in lower_function_source:
                    function_candidates.append({"File": str(source_path.relative_to(environment_package_path)), "Function": current_function, "Start Line": int(current_start), "Source Lines": int(len(current_lines)), "Matched Terms": ", ".join(matched_terms)})

            current_function = function_match.group(1)
            current_start = line_number
            current_lines = [line]
            continue

        if current_function is not None:
            if line and not line.startswith((" ", "\t")):
                function_source = "\n".join(current_lines)
                lower_function_source = function_source.lower()
                matched_terms = sorted({term for term in search_terms if term in lower_function_source})

                if "price" in lower_function_source or "prices" in lower_function_source:
                    function_candidates.append({"File": str(source_path.relative_to(environment_package_path)), "Function": current_function, "Start Line": int(current_start), "Source Lines": int(len(current_lines)), "Matched Terms": ", ".join(matched_terms)})

                current_function = None
                current_start = None
                current_lines = []
            else:
                current_lines.append(line)

    if current_function is not None:
        function_source = "\n".join(current_lines)
        lower_function_source = function_source.lower()
        matched_terms = sorted({term for term in search_terms if term in lower_function_source})

        if "price" in lower_function_source or "prices" in lower_function_source:
            function_candidates.append({"File": str(source_path.relative_to(environment_package_path)), "Function": current_function, "Start Line": int(current_start), "Source Lines": int(len(current_lines)), "Matched Terms": ", ".join(matched_terms)})

pricing_function_df = pd.DataFrame(function_candidates)

constant_pattern = re.compile(r"^\s*([A-Z][A-Z0-9_]{2,})\s*=\s*(.+)$")
pricing_constants = []

for source_path in python_files:
    try:
        source_lines = source_path.read_text(encoding = "utf-8").splitlines()
    except Exception:
        continue

    for line_number, line in enumerate(source_lines, start = 1):
        constant_match = constant_pattern.match(line)

        if constant_match is None:
            continue

        constant_name = constant_match.group(1)
        constant_value = constant_match.group(2)
        lower_text = f"{constant_name} {constant_value}".lower()

        if any(term in lower_text for term in search_terms):
            pricing_constants.append({"File": str(source_path.relative_to(environment_package_path)), "Line": int(line_number), "Constant": constant_name, "Value": constant_value})

pricing_constants_df = pd.DataFrame(pricing_constants)

commodity_names = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]
commodity_source_rows = []

for commodity in commodity_names:
    match_count = int(pricing_source_df["Source"].str.contains(commodity, case = False, regex = False).sum())
    commodity_source_rows.append({"Commodity": commodity, "Source Matches": match_count})

commodity_source_coverage_df = pd.DataFrame(commodity_source_rows)
current_main_sha256 = hashlib.sha256(Path(main_path).read_bytes()).hexdigest()
main_unchanged = current_main_sha256 == main_sha256_before_analysis

pricing_discovery_summary_df = pd.DataFrame({"Metric": ["Kaggle Environments Version", "Python Source Files Scanned", "Pricing Related Source Matches", "Price Related Functions", "Pricing Related Constants", "Environment Source", "Main File Unchanged", "Analysis Mode"], "Value": [str(kaggle_environments.__version__), f"{len(python_files):,}", f"{len(pricing_source_df):,}", f"{len(pricing_function_df):,}", f"{len(pricing_constants_df):,}", str(environment_package_path), str(main_unchanged), "READ ONLY"]})

priority_source_display_df = pricing_source_df[pricing_source_df["Priority"] == 1].head(20).copy()
priority_source_display_df = priority_source_display_df.drop(columns = "Priority")

display(pricing_discovery_summary_df)
display(commodity_source_coverage_df)
display(priority_source_display_df)

if not pricing_function_df.empty:
    display(pricing_function_df)

if not pricing_constants_df.empty:
    display(pricing_constants_df)

## Loading the real pricing function

Imports the genuine `kaggriculture` module from the installed environment
package and pulls its `MARKET_PARAMS`, equilibrium inventory (`MARKET_I0`),
and price floor directly from source. Computes the real price curve for
every product across a wide range of inventory levels, so the notebook's
price analysis reflects the environment's actual formula rather than a
hand-reconstructed approximation.

In [ ]:
import contextlib, hashlib, io, logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Analyze the original commodity price curves without modifying main.py
logging.disable(logging.CRITICAL)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from kaggle_environments.envs.kaggriculture import kaggriculture as kg

logging.disable(logging.NOTSET)

required_objects = ["main_path", "main_sha256_before_analysis"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required notebook objects are missing: " + ", ".join(missing_objects))

market_params = kg.MARKET_PARAMS
market_i0 = int(kg.MARKET_I0)
price_floor = int(kg.PRICE_FLOOR)
products = list(kg.PRODUCTS)

if not isinstance(market_params, dict) or not market_params:
    raise RuntimeError("Kaggriculture MARKET_PARAMS could not be loaded.")

inventory_minimum = 0
inventory_maximum = market_i0 * 2
inventory_points = 401
inventory_grid = np.linspace(inventory_minimum, inventory_maximum, inventory_points).astype(int)
pricing_parameter_rows = []
price_curve_rows = []

for product in products:
    parameters = market_params.get(product, {})

    if not isinstance(parameters, dict):
        continue

    base_price = parameters.get("base")
    amplitude = parameters.get("amp")
    shape = parameters.get("f")

    pricing_parameter_rows.append({"Product": product, "Base Price": base_price, "Amplitude": amplitude, "Shape": str(shape), "Equilibrium Inventory": market_i0, "Price Floor": price_floor})

    for inventory in inventory_grid:
        price = kg.market_price(product, int(inventory), market_params)
        price_curve_rows.append({"product": product, "inventory": int(inventory), "price": int(price)})

pricing_parameters_df = pd.DataFrame(pricing_parameter_rows)
price_curves_df = pd.DataFrame(price_curve_rows)

if pricing_parameters_df.empty or price_curves_df.empty:
    raise RuntimeError("Commodity pricing analysis produced no records.")

reference_inventory_levels = sorted({0, market_i0 // 4, market_i0 // 2, market_i0, int(market_i0 * 1.5), market_i0 * 2})
reference_price_rows = []

for product in products:
    for inventory in reference_inventory_levels:
        reference_price_rows.append({"Product": product, "Inventory": int(inventory), "Price": int(kg.market_price(product, int(inventory), market_params))})

reference_prices_df = pd.DataFrame(reference_price_rows)
reference_price_matrix_df = reference_prices_df.pivot(index = "Product", columns = "Inventory", values = "Price").reindex(products)
reference_price_matrix_df.columns = [f"Inventory {int(column):,}" for column in reference_price_matrix_df.columns]
reference_price_matrix_df = reference_price_matrix_df.reset_index()

sns.set_theme(style = "whitegrid")
figure, axis = plt.subplots(figsize = (15, 9))
palette = sns.color_palette("husl", n_colors = len(products))

for product, color in zip(products, palette):
    product_curve = price_curves_df[price_curves_df["product"] == product]
    axis.plot(product_curve["inventory"], product_curve["price"], linewidth = 2.3, color = color, label = product)

axis.axvline(market_i0, color = "#343A40", linestyle = "--", linewidth = 1.6, alpha = 0.85, label = f"Equilibrium Inventory ({market_i0:,})")
axis.axhline(price_floor, color = "#6C757D", linestyle = ":", linewidth = 1.5, alpha = 0.8, label = f"Price Floor ({price_floor})")
axis.set_title("Kaggriculture Commodity Price Curves", fontsize = 17, pad = 15)
axis.set_xlabel("Market Inventory", fontsize = 11)
axis.set_ylabel("Market Price per Unit", fontsize = 11)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(alpha = 0.22)
axis.legend(frameon = False, bbox_to_anchor = (1.02, 1), loc = "upper left")

plt.tight_layout()
plt.show()

current_main_sha256 = hashlib.sha256(Path(main_path).read_bytes()).hexdigest()
main_unchanged = current_main_sha256 == main_sha256_before_analysis

commodity_price_summary_df = pd.DataFrame({"Metric": ["Products Analyzed", "Equilibrium Inventory", "Price Floor", "Minimum Inventory", "Maximum Inventory", "Inventory Evaluation Points", "Price Curve Records", "Pricing Source", "Main File Unchanged"], "Value": [f"{len(products):,}", f"{market_i0:,}", f"{price_floor:,}", f"{inventory_minimum:,}", f"{inventory_maximum:,}", f"{inventory_points:,}", f"{len(price_curves_df):,}", "ORIGINAL KAGGRICULTURE ENVIRONMENT", str(main_unchanged)]})

display(commodity_price_summary_df)
display(pricing_parameters_df)
display(reference_price_matrix_df)

## Commodity pricing reference table

Summarizes each product's pricing function shape and parameters, alongside
its real computed price at several reference inventory levels (0, 0.5x, 1x, 1.5x, and 2x equilibrium) — a ground-truth lookup
table for how aggressively each commodity's price moves with supply.

In [ ]:
import hashlib
from pathlib import Path

import pandas as pd

# Summarize the pricing function structure and reference behavior of every Kaggriculture commodity
required_objects = ["kg", "market_params", "market_i0", "price_floor", "products", "main_path", "main_sha256_before_analysis"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required commodity pricing objects are missing: " + ", ".join(missing_objects))

pricing_function_rows = []

for product in products:
    parameters = market_params.get(product, {})

    if not isinstance(parameters, dict):
        continue

    base_price = float(parameters.get("base", 0))
    amplitude = float(parameters.get("amp", 0))
    shape_object = parameters.get("f")
    shape_name = getattr(shape_object, "__name__", str(shape_object))
    price_inventory_0 = int(kg.market_price(product, 0, market_params))
    price_inventory_half = int(kg.market_price(product, market_i0 // 2, market_params))
    price_inventory_equilibrium = int(kg.market_price(product, market_i0, market_params))
    price_inventory_high = int(kg.market_price(product, int(market_i0 * 1.5), market_params))
    price_inventory_double = int(kg.market_price(product, market_i0 * 2, market_params))

    pricing_function_rows.append({"Commodity": product, "Shape": shape_name, "Base Price": base_price, "Amplitude": amplitude, "Price at 0 Inventory": price_inventory_0, "Price at 0.5 I0": price_inventory_half, "Price at I0": price_inventory_equilibrium, "Price at 1.5 I0": price_inventory_high, "Price at 2 I0": price_inventory_double})

commodity_pricing_function_df = pd.DataFrame(pricing_function_rows)

if commodity_pricing_function_df.empty:
    raise RuntimeError("Pricing function summary produced no records.")

pricing_shape_summary_df = commodity_pricing_function_df.groupby("Shape", as_index = False).agg(Commodities = ("Commodity", "count"), Mean_Base_Price = ("Base Price", "mean"), Mean_Amplitude = ("Amplitude", "mean"))
pricing_shape_summary_df = pricing_shape_summary_df.sort_values(["Commodities", "Shape"], ascending = [False, True]).reset_index(drop = True)
pricing_shape_summary_df.columns = ["Shape", "Commodities", "Mean Base Price", "Mean Amplitude"]

current_main_sha256 = hashlib.sha256(Path(main_path).read_bytes()).hexdigest()
main_unchanged = current_main_sha256 == main_sha256_before_analysis

pricing_function_summary_df = pd.DataFrame({"Metric": ["Commodities Analyzed", "Unique Pricing Shapes", "Equilibrium Inventory", "Price Floor", "Main File Unchanged", "Analysis Mode"], "Value": [f"{len(commodity_pricing_function_df):,}", f"{commodity_pricing_function_df['Shape'].nunique():,}", f"{market_i0:,}", f"{price_floor:,}", str(main_unchanged), "READ ONLY"]})

display(pricing_function_summary_df)
display(pricing_shape_summary_df)
display(commodity_pricing_function_df)

## Price sensitivity per commodity

Numerically estimates each product's local price slope around equilibrium
inventory, plus its overall min/max price range from the earlier price
curves. This quantifies which commodities are price-fragile (a small
oversupply crashes the price hard) versus resilient (price barely moves) —
directly informing how cautiously the agent should batch sells for each one.

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Measure commodity price sensitivity using the original Kaggriculture pricing function
required_objects = ["kg", "market_params", "market_i0", "price_floor", "products", "price_curves_df", "main_path", "main_sha256_before_analysis"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required commodity pricing objects are missing: " + ", ".join(missing_objects))

sensitivity_step = 100
sensitivity_rows = []

for product in products:
    parameters = market_params.get(product, {})
    base_price = float(parameters.get("base", 0))
    amplitude = float(parameters.get("amp", 0))
    shape_name = str(parameters.get("f", "UNKNOWN"))
    price_left = float(kg.market_price(product, market_i0 - sensitivity_step, market_params))
    price_center = float(kg.market_price(product, market_i0, market_params))
    price_right = float(kg.market_price(product, market_i0 + sensitivity_step, market_params))
    local_slope = (price_right - price_left) / (2 * sensitivity_step)
    left_slope = (price_center - price_left) / sensitivity_step
    right_slope = (price_right - price_center) / sensitivity_step
    product_curve = price_curves_df[price_curves_df["product"] == product].copy()
    product_curve = product_curve.sort_values("inventory").reset_index(drop = True)
    minimum_price = float(product_curve["price"].min())
    maximum_price = float(product_curve["price"].max())
    price_range = maximum_price - minimum_price
    inventory_changes = product_curve["inventory"].diff()
    price_changes = product_curve["price"].diff()
    valid_change_mask = inventory_changes.notna() & (inventory_changes != 0)
    discrete_slopes = price_changes[valid_change_mask] / inventory_changes[valid_change_mask]
    mean_absolute_slope = float(discrete_slopes.abs().mean()) if not discrete_slopes.empty else 0.0
    maximum_absolute_slope = float(discrete_slopes.abs().max()) if not discrete_slopes.empty else 0.0
    floor_rows = product_curve[product_curve["price"] <= price_floor]
    floor_inventory = int(floor_rows["inventory"].min()) if not floor_rows.empty else np.nan

    sensitivity_rows.append({"product": product, "shape": shape_name, "base_price": base_price, "amplitude": amplitude, "price_at_equilibrium": price_center, "local_slope": local_slope, "left_slope": left_slope, "right_slope": right_slope, "mean_absolute_slope": mean_absolute_slope, "maximum_absolute_slope": maximum_absolute_slope, "minimum_price": minimum_price, "maximum_price": maximum_price, "price_range": price_range, "floor_inventory": floor_inventory})

commodity_sensitivity_df = pd.DataFrame(sensitivity_rows)

if commodity_sensitivity_df.empty:
    raise RuntimeError("Commodity sensitivity analysis produced no records.")

commodity_sensitivity_df["absolute_local_slope"] = commodity_sensitivity_df["local_slope"].abs()
commodity_sensitivity_df = commodity_sensitivity_df.sort_values(["absolute_local_slope", "price_range"], ascending = [False, False]).reset_index(drop = True)
commodity_sensitivity_df["sensitivity_rank"] = np.arange(1, len(commodity_sensitivity_df) + 1)

plot_sensitivity_df = commodity_sensitivity_df.sort_values("absolute_local_slope", ascending = True).reset_index(drop = True)

sns.set_theme(style = "whitegrid")
figure, axis = plt.subplots(figsize = (12, 7))
palette = sns.color_palette("husl", n_colors = len(plot_sensitivity_df))
bars = axis.barh(plot_sensitivity_df["product"], plot_sensitivity_df["absolute_local_slope"], color = palette)
maximum_sensitivity = float(plot_sensitivity_df["absolute_local_slope"].max())

for bar, value in zip(bars, plot_sensitivity_df["absolute_local_slope"]):
    label_offset = maximum_sensitivity * 0.02 if maximum_sensitivity > 0 else 0.001
    axis.text(bar.get_width() + label_offset, bar.get_y() + bar.get_height() / 2, f"{value:.4f}", va = "center", fontsize = 10)

axis.set_title("Commodity Price Sensitivity Near Market Equilibrium", fontsize = 16, pad = 15)
axis.set_xlabel(f"Absolute Price Change per Inventory Unit Around I0 ± {sensitivity_step}", fontsize = 11)
axis.set_ylabel("Commodity", fontsize = 11)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(axis = "x", alpha = 0.22)
axis.grid(axis = "y", visible = False)

plt.tight_layout()
plt.show()

figure, axis = plt.subplots(figsize = (13, 7))
scatter_palette = sns.color_palette("husl", n_colors = len(commodity_sensitivity_df))

for color, row in zip(scatter_palette, commodity_sensitivity_df.itertuples(index = False)):
    axis.scatter(row.price_range, row.absolute_local_slope, s = 140, color = color, alpha = 0.9)
    axis.annotate(row.product, (row.price_range, row.absolute_local_slope), xytext = (7, 5), textcoords = "offset points", fontsize = 9)

axis.set_title("Price Range vs Local Market Sensitivity", fontsize = 16, pad = 15)
axis.set_xlabel("Observed Price Range", fontsize = 11)
axis.set_ylabel("Absolute Local Price Sensitivity", fontsize = 11)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(alpha = 0.22)

plt.tight_layout()
plt.show()

most_sensitive = commodity_sensitivity_df.iloc[0]
least_sensitive = commodity_sensitivity_df.iloc[-1]
widest_range = commodity_sensitivity_df.loc[commodity_sensitivity_df["price_range"].idxmax()]
current_main_sha256 = hashlib.sha256(Path(main_path).read_bytes()).hexdigest()
main_unchanged = current_main_sha256 == main_sha256_before_analysis

commodity_sensitivity_summary_df = pd.DataFrame({"Metric": ["Sensitivity Reference Inventory", "Sensitivity Step", "Products Analyzed", "Most Sensitive Commodity", "Most Sensitive Local Slope", "Least Sensitive Commodity", "Least Sensitive Local Slope", "Widest Price Range Commodity", "Widest Observed Price Range", "Main File Unchanged"], "Value": [f"{market_i0:,}", f"{sensitivity_step:,}", f"{len(commodity_sensitivity_df):,}", str(most_sensitive["product"]), f"{most_sensitive['local_slope']:+.6f}", str(least_sensitive["product"]), f"{least_sensitive['local_slope']:+.6f}", str(widest_range["product"]), f"{widest_range['price_range']:,.0f}", str(main_unchanged)]})

commodity_sensitivity_display_df = commodity_sensitivity_df[["sensitivity_rank", "product", "shape", "price_at_equilibrium", "local_slope", "mean_absolute_slope", "price_range", "floor_inventory"]].copy()
commodity_sensitivity_display_df.columns = ["Rank", "Commodity", "Shape", "Price at I0", "Local Slope", "Mean Absolute Slope", "Price Range", "Floor Inventory"]
commodity_sensitivity_display_df["Floor Inventory"] = commodity_sensitivity_display_df["Floor Inventory"].apply(lambda value: "NOT REACHED" if pd.isna(value) else f"{int(value):,}")
commodity_sensitivity_display_df["Price at I0"] = commodity_sensitivity_display_df["Price at I0"].map(lambda value: f"{value:,.0f}")
commodity_sensitivity_display_df["Local Slope"] = commodity_sensitivity_display_df["Local Slope"].map(lambda value: f"{value:+.6f}")
commodity_sensitivity_display_df["Mean Absolute Slope"] = commodity_sensitivity_display_df["Mean Absolute Slope"].map(lambda value: f"{value:.6f}")
commodity_sensitivity_display_df["Price Range"] = commodity_sensitivity_display_df["Price Range"].map(lambda value: f"{value:,.0f}")

display(commodity_sensitivity_summary_df)
display(commodity_sensitivity_display_df)

## Checking trading behavior against price sensitivity

Joins the agent's actual `BUY_PRODUCT`/`SELL` trade counts against each
commodity's measured price sensitivity computed earlier, to verify the
embedded strategy trades fragile, high-impact commodities more carefully
than resilient ones — a consistency check between behavior and market
mechanics rather than an assumption.

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Connect the embedded trading plan with the original Kaggriculture market mechanics
required_objects = ["market_strategy_df", "commodity_sensitivity_df", "pricing_parameters_df", "products", "main_path", "main_sha256_before_analysis"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required trading and pricing objects are missing: " + ", ".join(missing_objects))

trading_actions = ["BUY_PRODUCT", "SELL"]
agent_trading_df = market_strategy_df[market_strategy_df["action"].isin(trading_actions)].copy()
agent_trading_df["product"] = agent_trading_df["arg1"].astype(str).str.strip().str.upper()
agent_trading_df = agent_trading_df[agent_trading_df["product"].isin(products)].reset_index(drop = True)

if agent_trading_df.empty:
    raise RuntimeError("No BUY_PRODUCT or SELL actions linked to market commodities were found.")

trade_count_df = agent_trading_df.groupby(["product", "action"]).size().unstack(fill_value = 0)

for action_name in trading_actions:
    if action_name not in trade_count_df.columns:
        trade_count_df[action_name] = 0

trade_count_df = trade_count_df.reindex(products, fill_value = 0)
trade_count_df["buy_count"] = trade_count_df["BUY_PRODUCT"].astype(int)
trade_count_df["sell_count"] = trade_count_df["SELL"].astype(int)
trade_count_df["total_trades"] = trade_count_df["buy_count"] + trade_count_df["sell_count"]
trade_count_df["net_trade_direction"] = trade_count_df["buy_count"] - trade_count_df["sell_count"]
trade_count_df = trade_count_df.reset_index()

pricing_link_df = commodity_sensitivity_df[["product", "shape", "base_price", "amplitude", "price_at_equilibrium", "local_slope", "absolute_local_slope", "mean_absolute_slope", "price_range", "sensitivity_rank"]].copy()
pricing_link_df = pricing_link_df.merge(trade_count_df[["product", "buy_count", "sell_count", "total_trades", "net_trade_direction"]], on = "product", how = "left")
pricing_link_df[["buy_count", "sell_count", "total_trades", "net_trade_direction"]] = pricing_link_df[["buy_count", "sell_count", "total_trades", "net_trade_direction"]].fillna(0).astype(int)
pricing_link_df["trade_share"] = pricing_link_df["total_trades"] / pricing_link_df["total_trades"].sum() if pricing_link_df["total_trades"].sum() else 0.0
pricing_link_df["sensitivity_exposure"] = pricing_link_df["total_trades"] * pricing_link_df["absolute_local_slope"]
pricing_link_df = pricing_link_df.sort_values(["total_trades", "absolute_local_slope"], ascending = [False, False]).reset_index(drop = True)

trade_matrix_df = pricing_link_df.set_index("product")[["buy_count", "sell_count"]]
trade_matrix_df.columns = ["Buy Product", "Sell"]

sns.set_theme(style = "white")
figure, axis = plt.subplots(figsize = (9, 7))

sns.heatmap(trade_matrix_df, cmap = "YlOrRd", annot = True, fmt = ".0f", linewidths = 0.6, linecolor = "white", cbar_kws = {"label": "Planned Trade Actions"}, ax = axis)

axis.set_title("Agent Trading Exposure by Commodity", fontsize = 16, pad = 15)
axis.set_xlabel("Trading Action", fontsize = 11)
axis.set_ylabel("Commodity", fontsize = 11)
axis.tick_params(axis = "x", rotation = 0)
axis.tick_params(axis = "y", rotation = 0)

plt.tight_layout()
plt.show()

active_trade_df = pricing_link_df[pricing_link_df["total_trades"] > 0].copy()

sns.set_theme(style = "whitegrid")
figure, axis = plt.subplots(figsize = (13, 8))

palette = sns.color_palette("husl", n_colors = len(active_trade_df))
maximum_trades = float(active_trade_df["total_trades"].max()) if not active_trade_df.empty else 1.0

for color, row in zip(palette, active_trade_df.itertuples(index = False)):
    bubble_size = 180 + 900 * row.total_trades / maximum_trades
    axis.scatter(row.absolute_local_slope, row.price_range, s = bubble_size, color = color, alpha = 0.78, edgecolor = "white", linewidth = 1.2)
    axis.annotate(f"{row.product}\n{int(row.total_trades)} trades", (row.absolute_local_slope, row.price_range), xytext = (7, 6), textcoords = "offset points", fontsize = 9)

axis.set_title("Trading Exposure vs Commodity Price Behavior", fontsize = 16, pad = 15)
axis.set_xlabel("Absolute Local Price Sensitivity", fontsize = 11)
axis.set_ylabel("Observed Price Range", fontsize = 11)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(alpha = 0.22)

plt.tight_layout()
plt.show()

daily_trade_df = agent_trading_df.groupby(["day", "action"]).size().unstack(fill_value = 0)

for action_name in trading_actions:
    if action_name not in daily_trade_df.columns:
        daily_trade_df[action_name] = 0

daily_trade_df = daily_trade_df.reindex(range(30), fill_value = 0)

figure, axis = plt.subplots(figsize = (14, 6))

axis.bar(daily_trade_df.index, daily_trade_df["BUY_PRODUCT"], color = "#4361EE", width = 0.78, label = "Buy Product")
axis.bar(daily_trade_df.index, daily_trade_df["SELL"], bottom = daily_trade_df["BUY_PRODUCT"], color = "#F72585", width = 0.78, label = "Sell")
axis.set_title("Planned Commodity Trading Activity Across Game Days", fontsize = 16, pad = 15)
axis.set_xlabel("Game Day", fontsize = 11)
axis.set_ylabel("Number of Trading Actions", fontsize = 11)
axis.set_xticks(daily_trade_df.index)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(axis = "y", alpha = 0.22)
axis.grid(axis = "x", visible = False)
axis.legend(frameon = False)

plt.tight_layout()
plt.show()

total_buys = int(pricing_link_df["buy_count"].sum())
total_sells = int(pricing_link_df["sell_count"].sum())
total_trades = int(pricing_link_df["total_trades"].sum())
traded_commodities = int((pricing_link_df["total_trades"] > 0).sum())
most_traded = pricing_link_df.loc[pricing_link_df["total_trades"].idxmax()]
highest_exposure = pricing_link_df.loc[pricing_link_df["sensitivity_exposure"].idxmax()]
current_main_sha256 = hashlib.sha256(Path(main_path).read_bytes()).hexdigest()
main_unchanged = current_main_sha256 == main_sha256_before_analysis

trading_summary_df = pd.DataFrame({"Metric": ["Total Planned Commodity Trades", "BUY_PRODUCT Actions", "SELL Actions", "Traded Commodities", "Most Traded Commodity", "Most Traded Commodity Actions", "Highest Sensitivity Exposure Commodity", "Highest Sensitivity Exposure Score", "Main File Unchanged"], "Value": [f"{total_trades:,}", f"{total_buys:,}", f"{total_sells:,}", f"{traded_commodities}/{len(products)}", str(most_traded["product"]), f"{int(most_traded['total_trades']):,}", str(highest_exposure["product"]), f"{highest_exposure['sensitivity_exposure']:.6f}", str(main_unchanged)]})

trading_profile_df = pricing_link_df[["product", "buy_count", "sell_count", "total_trades", "net_trade_direction", "trade_share", "shape", "price_at_equilibrium", "absolute_local_slope", "price_range"]].copy()
trading_profile_df["Direction"] = trading_profile_df["net_trade_direction"].apply(lambda value: "NET BUY" if value > 0 else "NET SELL" if value < 0 else "BALANCED")
trading_profile_df = trading_profile_df.drop(columns = "net_trade_direction")
trading_profile_df.columns = ["Commodity", "Buy", "Sell", "Total", "Trade Share", "Shape", "Price at I0", "Sensitivity", "Price Range", "Direction"]
trading_profile_df["Trade Share"] = trading_profile_df["Trade Share"].map(lambda value: f"{value:.2%}")
trading_profile_df["Price at I0"] = trading_profile_df["Price at I0"].map(lambda value: f"{value:,.0f}")
trading_profile_df["Sensitivity"] = trading_profile_df["Sensitivity"].map(lambda value: f"{value:.6f}")
trading_profile_df["Price Range"] = trading_profile_df["Price Range"].map(lambda value: f"{value:,.0f}")

display(trading_summary_df)
display(trading_profile_df)

## Auditing the adaptive runtime layer

Inspects the six adaptive functions layered on top of the frozen route
(`_weed_repair_action`, `_repay_shift`, `_rank_sell_slots`, `_preempt_shift`,
`_terminal_liquidation`, `_align_hands`), pulling their source code and a
plain-language description of what each one protects against into a single
audit table — documentation for why the agent is expected to survive board
conditions that differ from the one the route was recorded on.

In [ ]:
import ast, hashlib, inspect, textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Inspect the adaptive runtime components layered on top of the embedded action route
required_objects = ["agent_module", "main_path", "main_sha256_before_analysis"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required agent inspection objects are missing: " + ", ".join(missing_objects))

adaptation_functions = ["_weed_repair_action", "_repay_shift", "_rank_sell_slots", "_preempt_shift", "_terminal_liquidation", "_align_hands"]

adaptation_purposes = {"_weed_repair_action": "Repairs planned farming behavior when weed conditions require intervention", "_repay_shift": "Returns quantities previously shifted away from the original plan", "_rank_sell_slots": "Ranks planned selling opportunities using runtime market information", "_preempt_shift": "Moves selected actions earlier when current conditions justify preemption", "_terminal_liquidation": "Adjusts late game behavior to liquidate resources before the episode ends", "_align_hands": "Aligns planned worker actions with the currently available hand structure"}

adaptation_rows = []

for function_name in adaptation_functions:
    function_object = getattr(agent_module, function_name, None)

    if not callable(function_object):
        adaptation_rows.append({"function": function_name, "available": False, "source_lines": 0, "parameters": 0, "function_calls": 0, "branches": 0, "loops": 0, "returns": 0, "complexity_score": 0.0, "purpose": adaptation_purposes.get(function_name, "")})
        continue

    source_text = textwrap.dedent(inspect.getsource(function_object))
    function_tree = ast.parse(source_text)
    source_lines = len([line for line in source_text.splitlines() if line.strip()])
    parameter_count = len(inspect.signature(function_object).parameters)
    function_calls = sum(isinstance(node, ast.Call) for node in ast.walk(function_tree))
    branches = sum(isinstance(node, ast.If) for node in ast.walk(function_tree))
    loops = sum(isinstance(node, (ast.For, ast.While)) for node in ast.walk(function_tree))
    returns = sum(isinstance(node, ast.Return) for node in ast.walk(function_tree))
    complexity_score = branches + loops + function_calls * 0.25

    adaptation_rows.append({"function": function_name, "available": True, "source_lines": int(source_lines), "parameters": int(parameter_count), "function_calls": int(function_calls), "branches": int(branches), "loops": int(loops), "returns": int(returns), "complexity_score": float(complexity_score), "purpose": adaptation_purposes.get(function_name, "")})

runtime_adaptation_df = pd.DataFrame(adaptation_rows)
available_adaptation_df = runtime_adaptation_df[runtime_adaptation_df["available"]].copy()

if available_adaptation_df.empty:
    raise RuntimeError("No runtime adaptation functions could be inspected.")

available_adaptation_df["display_name"] = available_adaptation_df["function"].str.lstrip("_").str.replace("_", " ").str.title()
available_adaptation_df = available_adaptation_df.sort_values("complexity_score", ascending = True).reset_index(drop = True)

sns.set_theme(style = "whitegrid")
figure, axis = plt.subplots(figsize = (13, 7))
palette = sns.color_palette("husl", n_colors = len(available_adaptation_df))
bars = axis.barh(available_adaptation_df["display_name"], available_adaptation_df["complexity_score"], color = palette)
maximum_complexity = float(available_adaptation_df["complexity_score"].max())

for bar, value in zip(bars, available_adaptation_df["complexity_score"]):
    offset = maximum_complexity * 0.02 if maximum_complexity > 0 else 0.1
    axis.text(bar.get_width() + offset, bar.get_y() + bar.get_height() / 2, f"{value:.2f}", va = "center", fontsize = 10)

axis.set_title("Runtime Adaptation Component Complexity", fontsize = 16, pad = 15)
axis.set_xlabel("Structural Complexity Score", fontsize = 11)
axis.set_ylabel("Adaptive Component", fontsize = 11)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)
axis.grid(axis = "x", alpha = 0.22)
axis.grid(axis = "y", visible = False)

plt.tight_layout()
plt.show()

architecture_matrix_df = available_adaptation_df.set_index("display_name")[["parameters", "function_calls", "branches", "loops", "returns"]]

figure, axis = plt.subplots(figsize = (11, 6))

sns.heatmap(architecture_matrix_df, cmap = "viridis", annot = True, fmt = ".0f", linewidths = 0.5, linecolor = "white", cbar_kws = {"label": "Structural Count"}, ax = axis)

axis.set_title("Runtime Adaptation Architecture Matrix", fontsize = 16, pad = 15)
axis.set_xlabel("Source Structure", fontsize = 11)
axis.set_ylabel("Adaptive Component", fontsize = 11)
axis.tick_params(axis = "x", rotation = 25)
axis.tick_params(axis = "y", rotation = 0)

plt.tight_layout()
plt.show()

most_complex = available_adaptation_df.loc[available_adaptation_df["complexity_score"].idxmax()]
largest_component = available_adaptation_df.loc[available_adaptation_df["source_lines"].idxmax()]
total_source_lines = int(available_adaptation_df["source_lines"].sum())
total_branches = int(available_adaptation_df["branches"].sum())
total_loops = int(available_adaptation_df["loops"].sum())
total_function_calls = int(available_adaptation_df["function_calls"].sum())
current_main_sha256 = hashlib.sha256(Path(main_path).read_bytes()).hexdigest()
main_unchanged = current_main_sha256 == main_sha256_before_analysis

runtime_architecture_summary_df = pd.DataFrame({"Metric": ["Expected Adaptive Components", "Available Adaptive Components", "Adaptive Source Lines", "Conditional Branches", "Loops", "Function Calls", "Most Complex Component", "Most Complex Score", "Largest Component", "Largest Component Lines", "Main File Unchanged"], "Value": [f"{len(adaptation_functions):,}", f"{len(available_adaptation_df):,}", f"{total_source_lines:,}", f"{total_branches:,}", f"{total_loops:,}", f"{total_function_calls:,}", str(most_complex["function"]), f"{most_complex['complexity_score']:.2f}", str(largest_component["function"]), f"{int(largest_component['source_lines']):,}", str(main_unchanged)]})

runtime_architecture_display_df = runtime_adaptation_df.copy()
runtime_architecture_display_df["Status"] = runtime_architecture_display_df["available"].map({True: "AVAILABLE", False: "NOT FOUND"})
runtime_architecture_display_df = runtime_architecture_display_df[["function", "Status", "source_lines", "parameters", "function_calls", "branches", "loops", "returns", "complexity_score", "purpose"]]
runtime_architecture_display_df.columns = ["Function", "Status", "Lines", "Parameters", "Calls", "Branches", "Loops", "Returns", "Complexity", "Purpose"]
runtime_architecture_display_df["Complexity"] = runtime_architecture_display_df["Complexity"].map(lambda value: f"{value:.2f}")

display(runtime_architecture_summary_df)
display(runtime_architecture_display_df)

## Combined strategy dashboard

A single 2x2 figure combining the daily activity trend, top action
frequencies, commodity price sensitivity, and adaptive-layer summary into
one executive-level view of the whole agent — activity patterns, market
behavior, and defensive architecture side by side.

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Summarize the strategy market mechanics trading behavior and adaptive architecture
required_objects = ["daily_activity_df", "combined_action_df", "commodity_sensitivity_df", "pricing_link_df", "runtime_adaptation_df", "main_path", "main_sha256_before_analysis"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise RuntimeError("Required dashboard objects are missing: " + ", ".join(missing_objects))

sns.set_theme(style = "whitegrid")
figure, axes = plt.subplots(2, 2, figsize = (18, 13))
figure.suptitle("Kaggriculture Adaptive Agent Strategy Dashboard", fontsize = 21, fontweight = "bold", y = 0.99)

activity_axis = axes[0, 0]

activity_axis.plot(daily_activity_df["day"], daily_activity_df["farmer_actions"], color = "#E63946", marker = "o", linewidth = 2.2, markersize = 4, label = "Farmer")
activity_axis.plot(daily_activity_df["day"], daily_activity_df["hand_actions"], color = "#4361EE", marker = "s", linewidth = 2.2, markersize = 4, label = "Hands")
activity_axis.plot(daily_activity_df["day"], daily_activity_df["market_actions"], color = "#F4A261", marker = "^", linewidth = 2.2, markersize = 4, label = "Market")
activity_axis.plot(daily_activity_df["day"], daily_activity_df["total_actions"], color = "#6C757D", linestyle = "--", linewidth = 1.6, alpha = 0.8, label = "Total")
activity_axis.set_title("Daily Strategy Activity", fontsize = 14, pad = 10)
activity_axis.set_xlabel("Game Day")
activity_axis.set_ylabel("Active Actions")
activity_axis.spines["top"].set_visible(False)
activity_axis.spines["right"].set_visible(False)
activity_axis.grid(alpha = 0.2)
activity_axis.legend(frameon = False, ncol = 2)

action_axis = axes[0, 1]
dashboard_top_actions_df = combined_action_df.head(min(10, len(combined_action_df))).copy()
dashboard_top_actions_df = dashboard_top_actions_df.sort_values("Count", ascending = True).reset_index(drop = True)
action_palette = sns.color_palette("husl", n_colors = len(dashboard_top_actions_df))
action_bars = action_axis.barh(dashboard_top_actions_df["Action"], dashboard_top_actions_df["Count"], color = action_palette)
maximum_action_count = float(dashboard_top_actions_df["Count"].max()) if not dashboard_top_actions_df.empty else 1.0

for bar, count in zip(action_bars, dashboard_top_actions_df["Count"]):
    action_axis.text(bar.get_width() + maximum_action_count * 0.02, bar.get_y() + bar.get_height() / 2, f"{int(count):,}", va = "center", fontsize = 9)

action_axis.set_title("Top Embedded Actions", fontsize = 14, pad = 10)
action_axis.set_xlabel("Frequency")
action_axis.set_ylabel("Action")
action_axis.spines["top"].set_visible(False)
action_axis.spines["right"].set_visible(False)
action_axis.grid(axis = "x", alpha = 0.2)
action_axis.grid(axis = "y", visible = False)

sensitivity_axis = axes[1, 0]
dashboard_sensitivity_df = commodity_sensitivity_df.sort_values("absolute_local_slope", ascending = True).reset_index(drop = True)
sensitivity_palette = sns.color_palette("Spectral", n_colors = len(dashboard_sensitivity_df))
sensitivity_bars = sensitivity_axis.barh(dashboard_sensitivity_df["product"], dashboard_sensitivity_df["absolute_local_slope"], color = sensitivity_palette)
maximum_sensitivity = float(dashboard_sensitivity_df["absolute_local_slope"].max()) if not dashboard_sensitivity_df.empty else 1.0

for bar, value in zip(sensitivity_bars, dashboard_sensitivity_df["absolute_local_slope"]):
    sensitivity_axis.text(bar.get_width() + maximum_sensitivity * 0.02, bar.get_y() + bar.get_height() / 2, f"{value:.4f}", va = "center", fontsize = 9)

sensitivity_axis.set_title("Commodity Price Sensitivity", fontsize = 14, pad = 10)
sensitivity_axis.set_xlabel("Absolute Local Sensitivity")
sensitivity_axis.set_ylabel("Commodity")
sensitivity_axis.spines["top"].set_visible(False)
sensitivity_axis.spines["right"].set_visible(False)
sensitivity_axis.grid(axis = "x", alpha = 0.2)
sensitivity_axis.grid(axis = "y", visible = False)

trade_axis = axes[1, 1]
dashboard_trade_df = pricing_link_df[pricing_link_df["total_trades"] > 0].copy()
dashboard_trade_df = dashboard_trade_df.sort_values("total_trades", ascending = True).reset_index(drop = True)

if not dashboard_trade_df.empty:
    trade_axis.barh(dashboard_trade_df["product"], dashboard_trade_df["buy_count"], color = "#4361EE", label = "Buy Product")
    trade_axis.barh(dashboard_trade_df["product"], dashboard_trade_df["sell_count"], left = dashboard_trade_df["buy_count"], color = "#F72585", label = "Sell")

    for index, row in dashboard_trade_df.iterrows():
        trade_axis.text(row["total_trades"] + 0.15, index, f"{int(row['total_trades'])}", va = "center", fontsize = 9)

    trade_axis.legend(frameon = False)

trade_axis.set_title("Commodity Trading Exposure", fontsize = 14, pad = 10)
trade_axis.set_xlabel("Planned Trade Actions")
trade_axis.set_ylabel("Commodity")
trade_axis.spines["top"].set_visible(False)
trade_axis.spines["right"].set_visible(False)
trade_axis.grid(axis = "x", alpha = 0.2)
trade_axis.grid(axis = "y", visible = False)

plt.tight_layout(rect = [0, 0, 1, 0.97])
plt.show()

total_strategy_steps = int(len(route_df)) if "route_df" in globals() else 0
total_active_actions = int(daily_activity_df["total_actions"].sum())
total_market_actions = int(daily_activity_df["market_actions"].sum())
total_planned_trades = int(pricing_link_df["total_trades"].sum())
adaptive_components = int(runtime_adaptation_df["available"].sum())
most_active_day_row = daily_activity_df.loc[daily_activity_df["total_actions"].idxmax()]
most_common_action = combined_action_df.iloc[0]
most_sensitive_commodity = commodity_sensitivity_df.iloc[0]
most_traded_commodity = pricing_link_df.loc[pricing_link_df["total_trades"].idxmax()]
current_main_sha256 = hashlib.sha256(Path(main_path).read_bytes()).hexdigest()
main_unchanged = current_main_sha256 == main_sha256_before_analysis

dashboard_summary_df = pd.DataFrame({"Metric": ["Embedded Strategy Steps", "Total Active Actions", "Total Market Actions", "Planned Commodity Trades", "Runtime Adaptive Components", "Peak Activity Day", "Peak Activity Actions", "Most Frequent Action", "Most Sensitive Commodity", "Most Traded Commodity", "Main File Unchanged"], "Value": [f"{total_strategy_steps:,}", f"{total_active_actions:,}", f"{total_market_actions:,}", f"{total_planned_trades:,}", f"{adaptive_components:,}", f"Day {int(most_active_day_row['day'])}", f"{int(most_active_day_row['total_actions']):,}", str(most_common_action["Action"]), str(most_sensitive_commodity["product"]), str(most_traded_commodity["product"]), str(main_unchanged)]})
display(dashboard_summary_df)

## Packaging the submission

Final integrity and packaging pass: compiles `main.py` to confirm it is
syntactically valid Python, computes size/line/function/import statistics
and a sha256 hash of the file, then bundles it into `submission.tar.gz` and
re-opens the archive to confirm it contains exactly one member —
`main.py` — before it is considered ready to submit.

In [ ]:
import ast, hashlib, py_compile, tarfile
from pathlib import Path

# Package the generated main.py and report submission diagnostics
main_path = Path("/kaggle/working/main.py")
submission_path = Path("/kaggle/working/submission.tar.gz")

if not main_path.exists():
    raise FileNotFoundError(f"main.py not found: {main_path}")

py_compile.compile(str(main_path), doraise = True)

main_bytes = main_path.read_bytes()
main_text = main_bytes.decode("utf-8")
main_lines = main_text.splitlines()
main_ast = ast.parse(main_text)

main_characters = len(main_text)
main_byte_count = len(main_bytes)
main_kib = main_byte_count / 1024
main_mib = main_byte_count / (1024 ** 2)
total_lines = len(main_lines)
blank_lines = sum(1 for line in main_lines if not line.strip())
comment_lines = sum(1 for line in main_lines if line.lstrip().startswith("#"))
code_lines = total_lines - blank_lines - comment_lines
function_count = sum(isinstance(node, ast.FunctionDef) for node in ast.walk(main_ast))
import_count = sum(isinstance(node, (ast.Import, ast.ImportFrom)) for node in ast.walk(main_ast))
average_characters_per_line = main_characters / total_lines if total_lines else 0.0
main_sha256 = hashlib.sha256(main_bytes).hexdigest()

with tarfile.open(submission_path, "w:gz") as archive:
    archive.add(main_path, arcname = "main.py")

with tarfile.open(submission_path, "r:gz") as archive:
    members = archive.getnames()

    if members != ["main.py"]:
        raise RuntimeError(f"Invalid submission members: {members}")

    packaged_file = archive.extractfile("main.py")

    if packaged_file is None:
        raise RuntimeError("Unable to read main.py from submission archive.")

    packaged_bytes = packaged_file.read()

submission_bytes = submission_path.read_bytes()
submission_byte_count = len(submission_bytes)
submission_kib = submission_byte_count / 1024
submission_mib = submission_byte_count / (1024 ** 2)
submission_sha256 = hashlib.sha256(submission_bytes).hexdigest()
packaged_integrity = packaged_bytes == main_bytes
archive_ratio = submission_byte_count / main_byte_count if main_byte_count else 0.0

if not packaged_integrity:
    raise RuntimeError("Packaged main.py does not match source main.py.")

print("Submission Packaging Complete")
print(f"Main Path: {main_path}")
print(f"Characters: {main_characters:,}")
print(f"UTF 8 Bytes: {main_byte_count:,}")
print(f"File Size KiB: {main_kib:,.2f}")
print(f"File Size MiB: {main_mib:,.4f}")
print(f"Total Lines: {total_lines:,}")
print(f"Code Lines: {code_lines:,}")
print(f"Comment Lines: {comment_lines:,}")
print(f"Blank Lines: {blank_lines:,}")
print(f"Functions: {function_count:,}")
print(f"Imports: {import_count:,}")
print(f"Average Characters per Line: {average_characters_per_line:,.2f}")
print(f"Main SHA256: {main_sha256}")
print("Syntax Validation: PASS")
print(f"Submission Path: {submission_path}")
print(f"Submission Size Bytes: {submission_byte_count:,}")
print(f"Submission Size KiB: {submission_kib:,.2f}")
print(f"Submission Size MiB: {submission_mib:,.4f}")
print(f"Archive to Main Size Ratio: {archive_ratio:.4f}")
print(f"Submission SHA256: {submission_sha256}")
print(f"Submission Members: {', '.join(members)}")
print(f"Submission Files: {len(members)}")
print("Submission Structure Validation: PASS")
print(f"Packaged Main Integrity: {packaged_integrity}")
print("FINAL VERDICT: READY TO SUBMIT")

## Summary and next steps

The packaged agent combines a strong recorded route with six targeted
repairs for the ways a frozen plan and a live board can diverge. The
inspection cells above confirm two things worth remembering when iterating
further: the route's trading behavior is already consistent with each
commodity's real price sensitivity, and every adaptive safety function is
present and wired into the final `main.py`.

### The highest-leverage direction: no single route can be optimal everywhere

A single embedded route has a hard ceiling, and it is not a theoretical
concern — it is measurable. Two strong players can reach the exact same
board state and disagree on the best action from it, which means any one
recorded route is, at best, a good approximation at states it happened to
pass through and an unverified guess everywhere else. The adaptive layer
in this agent repairs *divergence* (a weed where the route expected an
empty tile, a hand count that does not match) but it does not repair
*disagreement* — a state the route never actually saw, where its default
answer might just be wrong rather than merely mismatched.

The concrete next step: instrument how often the repair layer actually
fires across a wide set of live matches. If `_weed_repair_action` and the
shift/preempt logic trigger rarely, the single-route approach is already
close to its ceiling. If they trigger often, that is direct evidence the
live game diverges from the recorded route regularly enough to justify
embedding two or three routes from different strong episodes and picking
between them at runtime using cheap state features (own vs. opponent board
signature, day, cash position) — the same kind of comparison
`_clone_distance` already makes, just extended from "is this state close
enough to trust the route" to "which of several routes fits this state
best."

### Other directions worth exploring

- **Repair coverage** — `_weed_repair_action` only handles weeds blocking
  `PLANT`/`BUILD_PASTURE`. Logging every case where the live tile disagrees
  with what the route expected (not just weeds) would show whether other
  divergence types are common enough to earn their own repair function.
- **Preempt calibration** — `_PREEMPT_FRACTION`, `_PREEMPT_MAX_BATCH`, and
  `_PREEMPT_MAX_CLONE_DISTANCE` are fixed constants. The price-impact
  scoring already built for `_order_score` could size these dynamically per
  commodity instead, using each product's measured sensitivity rather than
  one flat threshold for all of them.
- **Endgame margin** — `_terminal_liquidation` triggers at a fixed step.
  Checking how much shed value, if any, is still left unsold right before
  that trigger point across real matches would confirm whether it fires
  early enough given how much the workforce has grown by the late game.